# data_audit_and_scientific_profiling_pipeline

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

warnings.filterwarnings("ignore")

FILE_PATH = "/kaggle/input/datasets/jocelyndumlao/iot-monitoring-of-water-quality-and-tilapia/IoT Monitoring Dataset of Water Quality and Tilapia (Oreochromis niloticus) Health in Aquaculture Ponds in Montería, Colombia (2024))/Data_Model_IoTMLCQ_2024.xlsx"
OUT_DIR = Path("/kaggle/working/results/01_data_audit_and_profiling")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_DPI = 300
plt.rcParams["figure.dpi"] = FIG_DPI
plt.rcParams["savefig.dpi"] = FIG_DPI
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["xtick.labelsize"] = 8
plt.rcParams["ytick.labelsize"] = 8
plt.rcParams["legend.fontsize"] = 8
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

saved_files = []

xls = pd.ExcelFile(FILE_PATH)
sheet_name = xls.sheet_names[0]
df = pd.read_excel(FILE_PATH, sheet_name=sheet_name)

datetime_cols = df.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns.tolist()
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in df.columns if c not in numeric_cols + datetime_cols]

timestamp_col = "Datetime" if "Datetime" in df.columns else (datetime_cols[0] if len(datetime_cols) > 0 else None)

def extract_unit(col):
    m = re.search(r"\((.*?)\)", col)
    if m:
        return m.group(1).strip()
    manual = {
        "Datetime": "timestamp",
        "Month": "calendar month",
        "Month_Num": "month index",
        "month_x": "month index",
        "month_y": "month index",
        "day": "day index / derived",
        "hour": "hour index / derived",
        "pH": "pH units",
        "ph": "scaled / transformed",
        "oxigeno_scaled": "scaled / transformed",
        "turbidez": "scaled / transformed",
        "Oxygenation Interventions": "count",
        "Corrective Interventions": "count",
        "Oxygenation Automatic": "category",
        "Corrective Measures": "category",
        "Thermal Risk Index": "category",
        "Low Oxygen Alert": "category",
        "Health Status": "category",
    }
    return manual.get(col, "")

def infer_role(col):
    c = col.lower()
    if col == timestamp_col:
        return "timestamp"
    if c in ["month", "month_num", "month_x", "month_y", "day", "hour"]:
        return "calendar/derived_time"
    if any(k in c for k in ["average fish weight", "survival rate", "disease occurrence", "health status"]):
        return "target_candidate"
    if any(k in c for k in ["oxygenation interventions", "corrective interventions", "oxygenation automatic", "corrective measures"]):
        return "intervention/control"
    if any(k in c for k in ["thermal risk index", "low oxygen alert", "oxigeno_scaled", "turbidez", "ph"]):
        return "engineered/alert"
    if any(k in c for k in ["temperature", "oxygen", "dissolved oxygen", "ph", "turbidity", "precipitation"]):
        return "scientific_feature"
    return "other"

def infer_description(col):
    desc = {
        "Datetime": "Observation timestamp for the monitoring record.",
        "Month": "Calendar month label associated with the observation.",
        "Average Fish Weight (g)": "Average tilapia body weight at the observation period.",
        "Survival Rate (%)": "Observed fish survival percentage.",
        "Disease Occurrence (Cases)": "Recorded disease cases or disease burden indicator.",
        "Temperature (°C)": "Observed water temperature.",
        "Dissolved Oxygen (mg/L)": "Observed dissolved oxygen concentration in water.",
        "pH": "Observed water acidity/alkalinity.",
        "Turbidity (NTU)": "Observed water turbidity.",
        "Month_Num": "Numeric month encoding.",
        "month_x": "Derived or duplicated month-related numeric feature.",
        "Oxygenation Interventions": "Count/indicator of oxygenation interventions.",
        "Corrective Interventions": "Count of corrective interventions applied.",
        "Average Temperature (°C)": "Average ambient or contextual temperature feature.",
        "High Temperature (°C)": "High ambient or contextual temperature feature.",
        "Low Temperature (°C)": "Low ambient or contextual temperature feature.",
        "Precipitation (inches)": "Precipitation-related environmental feature.",
        "month_y": "Derived or duplicated month-related numeric feature.",
        "day": "Derived day-related feature.",
        "hour": "Derived hour-related feature.",
        "oxigeno_scaled": "Scaled/transformed oxygen-related feature.",
        "ph": "Scaled/transformed pH-related feature.",
        "turbidez": "Scaled/transformed turbidity-related feature.",
        "Oxygenation Automatic": "Automatic oxygenation system status.",
        "Corrective Measures": "Whether corrective measures were applied.",
        "Thermal Risk Index": "Categorical thermal-risk indicator.",
        "Low Oxygen Alert": "Categorical low-oxygen warning indicator.",
        "Health Status": "Categorical fish health status outcome."
    }
    return desc.get(col, "Dataset variable requiring domain-specific interpretation.")

def plausible_range_rule(col):
    rules = {
        "Average Fish Weight (g)": (0, 2000),
        "Survival Rate (%)": (0, 100),
        "Disease Occurrence (Cases)": (0, 10000),
        "Temperature (°C)": (0, 40),
        "Dissolved Oxygen (mg/L)": (0, 20),
        "pH": (0, 14),
        "Turbidity (NTU)": (0, 1000),
        "Month_Num": (1, 12),
        "month_x": (1, 12),
        "month_y": (1, 12),
        "Oxygenation Interventions": (0, 1),
        "Corrective Interventions": (0, 100),
        "Average Temperature (°C)": (-10, 60),
        "High Temperature (°C)": (-10, 60),
        "Low Temperature (°C)": (-20, 60),
        "Precipitation (inches)": (0, 100),
        "day": (1, 31),
        "hour": (0, 23),
    }
    return rules.get(col, None)

def robust_outlier_stats(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return {
            "non_null_count": 0, "q1": np.nan, "median": np.nan, "q3": np.nan,
            "iqr": np.nan, "lower_fence": np.nan, "upper_fence": np.nan,
            "outlier_count": 0, "outlier_rate_pct": 0.0
        }
    q1 = s.quantile(0.25)
    med = s.quantile(0.50)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        lower = q1
        upper = q3
        outlier_mask = pd.Series(False, index=s.index)
    else:
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outlier_mask = (s < lower) | (s > upper)
    outlier_count = int(outlier_mask.sum())
    outlier_rate = 100 * outlier_count / len(s)
    return {
        "non_null_count": len(s),
        "q1": q1,
        "median": med,
        "q3": q3,
        "iqr": iqr,
        "lower_fence": lower,
        "upper_fence": upper,
        "outlier_count": outlier_count,
        "outlier_rate_pct": outlier_rate
    }

def save_figure(fig, filename):
    out_path = OUT_DIR / filename
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    saved_files.append(str(out_path))

def save_csv(df_out, filename):
    out_path = OUT_DIR / filename
    df_out.to_csv(out_path, index=False)
    saved_files.append(str(out_path))

def short_text(ax, x, y, s, **kwargs):
    ax.text(x, y, s, transform=ax.transAxes, **kwargs)

role_order = {
    "timestamp": 0,
    "calendar/derived_time": 1,
    "scientific_feature": 2,
    "intervention/control": 3,
    "engineered/alert": 4,
    "target_candidate": 5,
    "other": 6
}

data_dict_rows = []
for col in df.columns:
    data_dict_rows.append({
        "variable_name": col,
        "data_type": str(df[col].dtype),
        "role": infer_role(col),
        "unit": extract_unit(col),
        "short_description": infer_description(col)
    })

table01 = pd.DataFrame(data_dict_rows)
table01["role_order"] = table01["role"].map(role_order)
table01["var_order"] = np.arange(len(table01))
table01 = table01.sort_values(["role_order", "var_order"]).drop(columns=["role_order", "var_order"]).reset_index(drop=True)
save_csv(table01, "table01_data_dictionary.csv")

table02 = pd.DataFrame({
    "variable_name": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100)
}).sort_values(["missing_pct", "missing_count", "variable_name"], ascending=[False, False, True]).reset_index(drop=True)
save_csv(table02, "table02_missing_value_summary.csv")

timestamp_audit = []
total_rows = len(df)
exact_duplicate_rows = int(df.duplicated().sum())

timestamp_audit.append({
    "audit_item": "total_rows",
    "value": total_rows,
    "notes": "Total observations in the dataset."
})
timestamp_audit.append({
    "audit_item": "total_columns",
    "value": df.shape[1],
    "notes": "Total variables in the dataset."
})
timestamp_audit.append({
    "audit_item": "exact_duplicate_rows",
    "value": exact_duplicate_rows,
    "notes": "Rows duplicated across all columns."
})

if timestamp_col is not None:
    ts = pd.to_datetime(df[timestamp_col], errors="coerce")
    sort_monotonic = bool(ts.is_monotonic_increasing)
    duplicate_timestamps = int(ts.duplicated().sum())
    invalid_timestamps = int(ts.isna().sum())

    diffs = ts.sort_values().diff().dropna()
    diffs_minutes = diffs.dt.total_seconds() / 60.0 if len(diffs) > 0 else pd.Series(dtype=float)

    negative_steps = int((diffs_minutes < 0).sum()) if len(diffs_minutes) > 0 else 0
    zero_steps = int((diffs_minutes == 0).sum()) if len(diffs_minutes) > 0 else 0
    positive_steps = diffs_minutes[diffs_minutes > 0] if len(diffs_minutes) > 0 else pd.Series(dtype=float)

    median_gap = float(positive_steps.median()) if len(positive_steps) > 0 else np.nan
    min_gap = float(positive_steps.min()) if len(positive_steps) > 0 else np.nan
    max_gap = float(positive_steps.max()) if len(positive_steps) > 0 else np.nan
    unique_gap_count = int(positive_steps.nunique()) if len(positive_steps) > 0 else 0

    large_gap_threshold = (median_gap * 1.5) if pd.notna(median_gap) else np.nan
    large_gap_count = int((positive_steps > large_gap_threshold).sum()) if pd.notna(large_gap_threshold) else 0

    timestamp_audit.extend([
        {"audit_item": "timestamp_column", "value": timestamp_col, "notes": "Primary timestamp column used for audit."},
        {"audit_item": "invalid_timestamps", "value": invalid_timestamps, "notes": "Rows with invalid or non-parsable timestamps."},
        {"audit_item": "duplicate_timestamps", "value": duplicate_timestamps, "notes": "Repeated timestamps in the primary timestamp column."},
        {"audit_item": "is_monotonic_increasing", "value": sort_monotonic, "notes": "Whether timestamps are already sorted in ascending order."},
        {"audit_item": "start_timestamp", "value": str(ts.min()), "notes": "Earliest valid timestamp."},
        {"audit_item": "end_timestamp", "value": str(ts.max()), "notes": "Latest valid timestamp."},
        {"audit_item": "median_gap_minutes", "value": median_gap, "notes": "Median positive time gap between sorted timestamps."},
        {"audit_item": "min_gap_minutes", "value": min_gap, "notes": "Minimum positive time gap between sorted timestamps."},
        {"audit_item": "max_gap_minutes", "value": max_gap, "notes": "Maximum positive time gap between sorted timestamps."},
        {"audit_item": "unique_positive_gap_count", "value": unique_gap_count, "notes": "Number of unique positive gap sizes."},
        {"audit_item": "large_gap_count", "value": large_gap_count, "notes": "Count of gaps larger than 1.5x the median gap."},
        {"audit_item": "negative_time_steps", "value": negative_steps, "notes": "Negative gaps after sorting; should usually be zero."},
        {"audit_item": "zero_time_steps", "value": zero_steps, "notes": "Zero-length steps after sorting; often caused by duplicated timestamps."},
    ])
else:
    timestamp_audit.append({
        "audit_item": "timestamp_column",
        "value": "Not found",
        "notes": "No datetime column available for timestamp consistency audit."
    })

table03 = pd.DataFrame(timestamp_audit)
save_csv(table03, "table03_duplicate_and_timestamp_audit.csv")

outlier_rows = []
for col in numeric_cols:
    stats = robust_outlier_stats(df[col])
    outlier_rows.append({
        "variable_name": col,
        **stats
    })

table04 = pd.DataFrame(outlier_rows).sort_values(
    ["outlier_rate_pct", "outlier_count", "variable_name"],
    ascending=[False, False, True]
).reset_index(drop=True)
save_csv(table04, "table04_outlier_summary.csv")

validity_rows = []

for col in df.columns:
    series = df[col]
    dtype_str = str(series.dtype)

    if pd.api.types.is_numeric_dtype(series):
        observed_min = series.min(skipna=True)
        observed_max = series.max(skipna=True)
        rule = plausible_range_rule(col)

        if rule is not None:
            lo, hi = rule
            invalid_count = int(((series < lo) | (series > hi)).sum())
            plausible_check = "PASS" if invalid_count == 0 else "FLAG"
            comment = f"Plausible numeric range checked against [{lo}, {hi}]."
        else:
            invalid_count = 0
            plausible_check = "NO_STRICT_RULE"
            comment = "No strict domain range rule applied; requires domain review."
        
        if series.nunique(dropna=True) == 1:
            comment += " Variable is constant."
        elif series.nunique(dropna=True) <= 2:
            comment += " Very low variability."
        
        validity_rows.append({
            "variable_name": col,
            "data_type": dtype_str,
            "observed_min": observed_min,
            "observed_max": observed_max,
            "plausible_range": str(rule) if rule is not None else "",
            "plausible_range_check": plausible_check,
            "invalid_value_count": invalid_count,
            "comments": comment
        })

    elif pd.api.types.is_datetime64_any_dtype(series):
        invalid_count = int(pd.to_datetime(series, errors="coerce").isna().sum())
        validity_rows.append({
            "variable_name": col,
            "data_type": dtype_str,
            "observed_min": str(series.min()),
            "observed_max": str(series.max()),
            "plausible_range": "Chronological timestamps",
            "plausible_range_check": "PASS" if invalid_count == 0 else "FLAG",
            "invalid_value_count": invalid_count,
            "comments": "Datetime validity checked using parsability."
        })

    else:
        nunique = int(series.nunique(dropna=True))
        cats = sorted(series.dropna().astype(str).unique().tolist())
        preview = ", ".join(cats[:6]) + (" ..." if len(cats) > 6 else "")
        validity_rows.append({
            "variable_name": col,
            "data_type": dtype_str,
            "observed_min": "",
            "observed_max": "",
            "plausible_range": "Categorical / domain-specific",
            "plausible_range_check": "REVIEW",
            "invalid_value_count": 0,
            "comments": f"{nunique} unique categories. Example values: {preview}"
        })

table05 = pd.DataFrame(validity_rows)
save_csv(table05, "table05_range_and_validity_checks.csv")

dtype_counts = pd.Series({
    "Numeric": len(numeric_cols),
    "Categorical": len(categorical_cols),
    "Datetime": len(datetime_cols)
})

role_counts = table01["role"].value_counts().sort_values(ascending=True)

if timestamp_col is not None:
    ts = pd.to_datetime(df[timestamp_col], errors="coerce")
    start_ts = str(ts.min())[:19]
    end_ts = str(ts.max())[:19]
else:
    start_ts, end_ts = "N/A", "N/A"

fig = plt.figure(figsize=(13, 6), constrained_layout=True)
gs = fig.add_gridspec(1, 3, width_ratios=[1.25, 1.0, 1.2])

ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[0, 1])
ax2 = fig.add_subplot(gs[0, 2])

ax0.axis("off")
summary_lines = [
    "Dataset Overview",
    f"Rows: {df.shape[0]:,}",
    f"Columns: {df.shape[1]}",
    f"Primary sheet: {sheet_name}",
    f"Timestamp column: {timestamp_col if timestamp_col else 'Not found'}",
    f"Time coverage: {start_ts} to {end_ts}",
    f"Exact duplicate rows: {exact_duplicate_rows}",
    f"Variables with any missing values: {(table02['missing_count'] > 0).sum()}",
    f"Numeric variables: {len(numeric_cols)}",
    f"Categorical variables: {len(categorical_cols)}",
    f"Datetime variables: {len(datetime_cols)}",
]
for i, line in enumerate(summary_lines):
    short_text(
        ax0, 0.02, 0.95 - i * 0.085, line,
        ha="left", va="top",
        fontsize=11 if i == 0 else 10,
        fontweight="bold" if i == 0 else None
    )

ax1.barh(dtype_counts.index, dtype_counts.values)
ax1.set_title("Data-Type Composition")
ax1.set_xlabel("Count")
for y, v in enumerate(dtype_counts.values):
    ax1.text(v + 0.1, y, str(v), va="center", fontsize=9)

ax2.barh(role_counts.index, role_counts.values)
ax2.set_title("Variable Role Composition")
ax2.set_xlabel("Count")
for y, v in enumerate(role_counts.values):
    ax2.text(v + 0.1, y, str(v), va="center", fontsize=9)

fig.suptitle("Dataset Dimensions, Structure, and Variable Composition", fontsize=13, fontweight="bold")
save_figure(fig, "fig01_dataset_overview.png")

miss_plot = table02.sort_values(["missing_pct", "variable_name"], ascending=[True, True])

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
ax.barh(miss_plot["variable_name"], miss_plot["missing_pct"])
ax.set_title("Variable-Wise Missingness Profile", fontweight="bold")
ax.set_xlabel("Missing values (%)")
ax.set_ylabel("Variable")
ax.xaxis.set_major_locator(MaxNLocator(integer=False))
for i, v in enumerate(miss_plot["missing_pct"]):
    ax.text(v + 0.05, i, f"{v:.1f}%", va="center", fontsize=8)

if miss_plot["missing_pct"].max() == 0:
    ax.text(0.5, 1.02, "No missing values detected in any variable.", transform=ax.transAxes,
            ha="center", va="bottom", fontsize=10, fontweight="bold")
save_figure(fig, "fig02_missingness_profile.png")

out_plot = table04.sort_values(["outlier_rate_pct", "variable_name"], ascending=[True, True])

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
ax.barh(out_plot["variable_name"], out_plot["outlier_rate_pct"])
ax.set_title("Robust Outlier Rate by Numeric Variable (IQR Method)", fontweight="bold")
ax.set_xlabel("Outlier rate (%)")
ax.set_ylabel("Numeric variable")
for i, v in enumerate(out_plot["outlier_rate_pct"]):
    ax.text(v + 0.05, i, f"{v:.2f}%", va="center", fontsize=8)
save_figure(fig, "fig03_outlier_profile.png")

priority_distribution_vars = [
    "Temperature (°C)",
    "Dissolved Oxygen (mg/L)",
    "pH",
    "Turbidity (NTU)",
    "Average Fish Weight (g)",
    "Survival Rate (%)"
]
dist_vars = [c for c in priority_distribution_vars if c in df.columns]
if len(dist_vars) < 6:
    remaining = [c for c in numeric_cols if c not in dist_vars]
    dist_vars += remaining[:(6 - len(dist_vars))]
dist_vars = dist_vars[:6]

fig, axes = plt.subplots(2, 3, figsize=(14, 7), constrained_layout=True)
axes = axes.flatten()

for ax, col in zip(axes, dist_vars):
    s = pd.to_numeric(df[col], errors="coerce").dropna()
    ax.hist(s, bins=25)
    ax.axvline(s.mean(), linestyle="--", linewidth=1.5, label=f"Mean = {s.mean():.2f}")
    ax.set_title(col)
    ax.set_xlabel("Value")
    ax.set_ylabel("Frequency")
    ax.legend(frameon=False)

for j in range(len(dist_vars), len(axes)):
    axes[j].axis("off")

fig.suptitle("Distribution Overview of Key Scientific Variables", fontsize=13, fontweight="bold")
save_figure(fig, "fig04_variable_distribution_panel.png")

fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
axes = axes.flatten()

if "Average Fish Weight (g)" in df.columns:
    s = pd.to_numeric(df["Average Fish Weight (g)"], errors="coerce").dropna()
    axes[0].hist(s, bins=25)
    axes[0].set_title("Average Fish Weight (g)")
    axes[0].set_xlabel("g")
    axes[0].set_ylabel("Frequency")
else:
    axes[0].axis("off")

if "Survival Rate (%)" in df.columns:
    s = pd.to_numeric(df["Survival Rate (%)"], errors="coerce").dropna()
    axes[1].hist(s, bins=25)
    axes[1].set_title("Survival Rate (%)")
    axes[1].set_xlabel("%")
    axes[1].set_ylabel("Frequency")
else:
    axes[1].axis("off")

if "Disease Occurrence (Cases)" in df.columns:
    counts = df["Disease Occurrence (Cases)"].value_counts().sort_index()
    axes[2].bar(counts.index.astype(str), counts.values)
    axes[2].set_title("Disease Occurrence (Cases)")
    axes[2].set_xlabel("Cases")
    axes[2].set_ylabel("Count")
else:
    axes[2].axis("off")

if "Health Status" in df.columns:
    counts = df["Health Status"].astype(str).value_counts()
    axes[3].bar(counts.index.astype(str), counts.values)
    axes[3].set_title("Health Status")
    axes[3].set_xlabel("Category")
    axes[3].set_ylabel("Count")
    axes[3].tick_params(axis="x", rotation=20)
else:
    axes[3].axis("off")

if {"Dissolved Oxygen (mg/L)", "Survival Rate (%)"}.issubset(df.columns):
    x = pd.to_numeric(df["Dissolved Oxygen (mg/L)"], errors="coerce")
    y = pd.to_numeric(df["Survival Rate (%)"], errors="coerce")
    valid = x.notna() & y.notna()
    x, y = x[valid], y[valid]
    axes[4].scatter(x, y, alpha=0.5, s=12)
    if x.nunique() > 1:
        m, b = np.polyfit(x, y, 1)
        xx = np.linspace(x.min(), x.max(), 100)
        axes[4].plot(xx, m * xx + b, linewidth=1.5)
    axes[4].set_title("Survival vs Dissolved Oxygen")
    axes[4].set_xlabel("Dissolved Oxygen (mg/L)")
    axes[4].set_ylabel("Survival Rate (%)")
else:
    axes[4].axis("off")

if {"Temperature (°C)", "Average Fish Weight (g)"}.issubset(df.columns):
    x = pd.to_numeric(df["Temperature (°C)"], errors="coerce")
    y = pd.to_numeric(df["Average Fish Weight (g)"], errors="coerce")
    valid = x.notna() & y.notna()
    x, y = x[valid], y[valid]
    axes[5].scatter(x, y, alpha=0.5, s=12)
    if x.nunique() > 1:
        m, b = np.polyfit(x, y, 1)
        xx = np.linspace(x.min(), x.max(), 100)
        axes[5].plot(xx, m * xx + b, linewidth=1.5)
    axes[5].set_title("Fish Weight vs Temperature")
    axes[5].set_xlabel("Temperature (°C)")
    axes[5].set_ylabel("Average Fish Weight (g)")
else:
    axes[5].axis("off")

fig.suptitle("Target and Response Variable Overview", fontsize=13, fontweight="bold")
save_figure(fig, "fig05_target_and_response_overview.png")

constant_cols = [c for c in df.columns if df[c].nunique(dropna=True) == 1]
low_var_cols = [c for c in df.columns if 1 < df[c].nunique(dropna=True) <= 2]

duplicate_column_pairs = []
for i, c1 in enumerate(df.columns):
    for c2 in df.columns[i+1:]:
        if df[c1].equals(df[c2]):
            duplicate_column_pairs.append((c1, c2))

top_outliers = table04[table04["outlier_count"] > 0].head(5)[["variable_name", "outlier_count", "outlier_rate_pct"]]

# multimodal_exploratory_analysis_pipeline

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

FILE_PATH = "/kaggle/input/datasets/jocelyndumlao/iot-monitoring-of-water-quality-and-tilapia/IoT Monitoring Dataset of Water Quality and Tilapia (Oreochromis niloticus) Health in Aquaculture Ponds in Montería, Colombia (2024))/Data_Model_IoTMLCQ_2024.xlsx"
OUT_DIR = Path("/kaggle/working/results/02_multimodal_exploratory_analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_DPI = 300
plt.rcParams["figure.dpi"] = FIG_DPI
plt.rcParams["savefig.dpi"] = FIG_DPI
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["xtick.labelsize"] = 8
plt.rcParams["ytick.labelsize"] = 8
plt.rcParams["legend.fontsize"] = 8
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

saved_files = []

xls = pd.ExcelFile(FILE_PATH)
sheet_name = xls.sheet_names[0]
df = pd.read_excel(FILE_PATH, sheet_name=sheet_name)

if "Datetime" in df.columns:
    df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")
    df = df.sort_values("Datetime").reset_index(drop=True)

water_vars = [
    "Temperature (°C)",
    "Dissolved Oxygen (mg/L)",
    "pH",
    "Turbidity (NTU)",
    "Precipitation (inches)",
]
tilapia_vars = [
    "Average Fish Weight (g)",
    "Survival Rate (%)",
    "Disease Occurrence (Cases)",
]
categorical_target_vars = ["Health Status"]

water_vars = [c for c in water_vars if c in df.columns]
tilapia_vars = [c for c in tilapia_vars if c in df.columns]
categorical_target_vars = [c for c in categorical_target_vars if c in df.columns]

core_vars = water_vars + tilapia_vars
numeric_core_vars = [c for c in core_vars if pd.api.types.is_numeric_dtype(df[c])]

corr_vars = [
    "Temperature (°C)",
    "Dissolved Oxygen (mg/L)",
    "pH",
    "Turbidity (NTU)",
    "Precipitation (inches)",
    "Average Fish Weight (g)",
    "Survival Rate (%)",
    "Disease Occurrence (Cases)",
]
corr_vars = [c for c in corr_vars if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]

if "Datetime" in df.columns:
    df["month_num_plot"] = df["Datetime"].dt.month
    df["month_name_plot"] = df["Datetime"].dt.strftime("%b")
else:
    if "Month" in df.columns:
        month_order_map = {
            "January": 1, "February": 2, "March": 3, "April": 4,
            "May": 5, "June": 6, "July": 7, "August": 8,
            "September": 9, "October": 10, "November": 11, "December": 12
        }
        df["month_num_plot"] = df["Month"].map(month_order_map)
        df["month_name_plot"] = df["Month"].str.slice(0, 3)
    else:
        raise ValueError("No usable Datetime or Month column found.")

month_order = (
    df[["month_num_plot", "month_name_plot"]]
    .drop_duplicates()
    .sort_values("month_num_plot")
    .reset_index(drop=True)
)

def save_csv(df_out, filename):
    out_path = OUT_DIR / filename
    df_out.to_csv(out_path, index=False)
    saved_files.append(str(out_path))

def save_figure(fig, filename):
    out_path = OUT_DIR / filename
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    saved_files.append(str(out_path))

def iqr(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    return x.quantile(0.75) - x.quantile(0.25)

def skewness(x):
    x = pd.Series(x).dropna()
    if len(x) < 3:
        return np.nan
    return x.skew()

def coeff_var(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    m = x.mean()
    s = x.std()
    if pd.isna(m) or m == 0:
        return np.nan
    return (s / abs(m)) * 100

def mad(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    med = x.median()
    return np.median(np.abs(x - med))

def add_trendline(ax, x, y):
    if len(x) < 2 or pd.Series(x).nunique() < 2:
        return np.nan
    m, b = np.polyfit(x, y, 1)
    xx = np.linspace(np.min(x), np.max(x), 100)
    ax.plot(xx, m * xx + b, linewidth=1.5)
    return np.corrcoef(x, y)[0, 1]

def qcut_labels(series):
    q1 = series.quantile(1 / 3)
    q2 = series.quantile(2 / 3)
    bins = [-np.inf, q1, q2, np.inf]
    labels = ["Low", "Medium", "High"]
    cat = pd.cut(series, bins=bins, labels=labels, include_lowest=True, duplicates="drop")
    return cat, q1, q2

table01_rows = []
for col in numeric_core_vars:
    s = pd.to_numeric(df[col], errors="coerce").dropna()
    table01_rows.append({
        "variable_name": col,
        "count": int(s.count()),
        "mean": s.mean(),
        "std": s.std(),
        "min": s.min(),
        "median": s.median(),
        "max": s.max(),
        "iqr": iqr(s),
        "skewness": skewness(s)
    })

table01 = pd.DataFrame(table01_rows)
table01["group"] = table01["variable_name"].apply(lambda x: "Water Quality" if x in water_vars else "Tilapia Outcome")
group_order = {"Water Quality": 0, "Tilapia Outcome": 1}
table01["group_order"] = table01["group"].map(group_order)
table01 = table01.sort_values(["group_order", "variable_name"]).drop(columns=["group_order"]).reset_index(drop=True)
save_csv(table01, "table01_descriptive_statistics.csv")

monthly_agg = df.groupby(["month_num_plot", "month_name_plot"]).agg(
    sample_count=("month_num_plot", "size"),
    **{
        f"{col}_mean": (col, "mean") for col in numeric_core_vars
    },
    **{
        f"{col}_std": (col, "std") for col in numeric_core_vars
    }
).reset_index()

monthly_agg = monthly_agg.sort_values("month_num_plot").reset_index(drop=True)
save_csv(monthly_agg, "table02_monthly_summary.csv")

corr_matrix = df[corr_vars].corr(numeric_only=True)

corr_rows = []
for i, c1 in enumerate(corr_vars):
    for c2 in corr_vars[i + 1:]:
        r = corr_matrix.loc[c1, c2]
        corr_rows.append({
            "variable_1": c1,
            "variable_2": c2,
            "correlation": r,
            "absolute_correlation": abs(r),
            "direction": "Positive" if r >= 0 else "Negative"
        })

corr_pairs = pd.DataFrame(corr_rows)
top_pos = corr_pairs[corr_pairs["correlation"] > 0].sort_values("correlation", ascending=False).head(6)
top_neg = corr_pairs[corr_pairs["correlation"] < 0].sort_values("correlation", ascending=True).head(6)
table03 = pd.concat([top_pos, top_neg], axis=0).reset_index(drop=True)
table03 = table03.sort_values(["direction", "absolute_correlation"], ascending=[True, False]).reset_index(drop=True)
save_csv(table03, "table03_top_correlated_pairs.csv")

condition_rows = []

stable_map = None
if "Health Status" in df.columns:
    stable_map = df["Health Status"].astype(str).str.lower().eq("stable").astype(int)

condition_source_vars = [c for c in ["Dissolved Oxygen (mg/L)", "Temperature (°C)", "pH", "Turbidity (NTU)"] if c in df.columns]

for var in condition_source_vars:
    s = pd.to_numeric(df[var], errors="coerce")
    cond_cat, q1, q2 = qcut_labels(s)

    tmp = df.copy()
    tmp["condition_level"] = cond_cat
    tmp = tmp.dropna(subset=["condition_level"])

    grouped = tmp.groupby("condition_level").agg(
        sample_count=("condition_level", "size"),
        average_fish_weight_mean=("Average Fish Weight (g)", "mean") if "Average Fish Weight (g)" in tmp.columns else ("condition_level", "size"),
        survival_rate_mean=("Survival Rate (%)", "mean") if "Survival Rate (%)" in tmp.columns else ("condition_level", "size"),
        disease_occurrence_mean=("Disease Occurrence (Cases)", "mean") if "Disease Occurrence (Cases)" in tmp.columns else ("condition_level", "size"),
    ).reset_index()

    if stable_map is not None:
        tmp["stable_binary"] = stable_map.loc[tmp.index]
        stable_pct = tmp.groupby("condition_level")["stable_binary"].mean().reset_index(name="stable_pct")
        grouped = grouped.merge(stable_pct, on="condition_level", how="left")
        grouped["stable_pct"] = grouped["stable_pct"] * 100
    else:
        grouped["stable_pct"] = np.nan

    threshold_text = f"Low ≤ {q1:.3f}; Medium ({q1:.3f}, {q2:.3f}]; High > {q2:.3f}"

    grouped.insert(0, "water_quality_variable", var)
    grouped.insert(2, "threshold_definition", threshold_text)
    condition_rows.append(grouped)

table04 = pd.concat(condition_rows, axis=0).reset_index(drop=True)
level_order = {"Low": 0, "Medium": 1, "High": 2}
table04["level_order"] = table04["condition_level"].astype(str).map(level_order)
table04 = table04.sort_values(["water_quality_variable", "level_order"]).drop(columns=["level_order"]).reset_index(drop=True)
save_csv(table04, "table04_target_condition_summary.csv")

table05_rows = []
for col in numeric_core_vars:
    s = pd.to_numeric(df[col], errors="coerce").dropna()
    table05_rows.append({
        "variable_name": col,
        "mean": s.mean(),
        "std": s.std(),
        "coefficient_of_variation_pct": coeff_var(s),
        "iqr": iqr(s),
        "mad": mad(s),
        "range": s.max() - s.min(),
    })

table05 = pd.DataFrame(table05_rows)
table05 = table05.sort_values("coefficient_of_variation_pct", ascending=False).reset_index(drop=True)
table05.insert(0, "variability_rank", np.arange(1, len(table05) + 1))
save_csv(table05, "table05_variability_ranking.csv")

trend_vars = [c for c in [
    "Temperature (°C)",
    "Dissolved Oxygen (mg/L)",
    "pH",
    "Turbidity (NTU)",
    "Average Fish Weight (g)",
    "Survival Rate (%)"
] if c in df.columns]

fig, axes = plt.subplots(3, 2, figsize=(14, 9), sharex=True, constrained_layout=True)
axes = axes.flatten()

for ax, col in zip(axes, trend_vars):
    series = pd.to_numeric(df[col], errors="coerce")
    if "Datetime" in df.columns:
        x = df["Datetime"]
    else:
        x = np.arange(len(df))
    rolling = series.rolling(window=24, min_periods=1).mean()
    ax.plot(x, rolling, linewidth=1.3)
    ax.set_title(col)
    ax.set_ylabel(col)
    ax.grid(alpha=0.2)

for j in range(len(trend_vars), len(axes)):
    axes[j].axis("off")

fig.suptitle("Temporal Trends of Major Water-Quality and Tilapia Variables", fontsize=13, fontweight="bold")
save_figure(fig, "fig01_temporal_trends.png")

period_vars = trend_vars[:6]

fig, axes = plt.subplots(3, 2, figsize=(14, 9), constrained_layout=True)
axes = axes.flatten()

for ax, col in zip(axes, period_vars):
    mean_col = f"{col}_mean"
    std_col = f"{col}_std"
    x = monthly_agg["month_name_plot"]
    y = monthly_agg[mean_col]
    yerr = monthly_agg[std_col].fillna(0)

    ax.errorbar(x, y, yerr=yerr, marker="o", linewidth=1.5, capsize=3)
    ax.set_title(col)
    ax.set_xlabel("Month")
    ax.set_ylabel(col)
    ax.grid(alpha=0.2)

for j in range(len(period_vars), len(axes)):
    axes[j].axis("off")

fig.suptitle("Monthly Behavior of Key Scientific Variables", fontsize=13, fontweight="bold")
save_figure(fig, "fig02_monthly_or_periodic_patterns.png")

heat_corr = df[corr_vars].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
im = ax.imshow(heat_corr.values, vmin=-1, vmax=1, aspect="auto")

ax.set_xticks(np.arange(len(corr_vars)))
ax.set_yticks(np.arange(len(corr_vars)))
ax.set_xticklabels(corr_vars, rotation=45, ha="right")
ax.set_yticklabels(corr_vars)

for i in range(len(corr_vars)):
    for j in range(len(corr_vars)):
        ax.text(j, i, f"{heat_corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Pearson correlation")
ax.set_title("Correlation Structure of Important Numeric Variables", fontweight="bold")
save_figure(fig, "fig03_correlation_heatmap.png")

relationship_pairs = [
    ("Dissolved Oxygen (mg/L)", "Survival Rate (%)"),
    ("Temperature (°C)", "Disease Occurrence (Cases)"),
    ("Turbidity (NTU)", "Average Fish Weight (g)"),
    ("pH", "Survival Rate (%)"),
]
relationship_pairs = [(x, y) for x, y in relationship_pairs if x in df.columns and y in df.columns]

fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
axes = axes.flatten()

for ax, (xcol, ycol) in zip(axes, relationship_pairs):
    x = pd.to_numeric(df[xcol], errors="coerce")
    y = pd.to_numeric(df[ycol], errors="coerce")
    valid = x.notna() & y.notna()
    x = x[valid].values
    y = y[valid].values

    ax.scatter(x, y, alpha=0.35, s=12)
    r = add_trendline(ax, x, y)
    ax.set_xlabel(xcol)
    ax.set_ylabel(ycol)
    ax.set_title(f"{ycol} vs {xcol}")
    ax.grid(alpha=0.2)
    if pd.notna(r):
        ax.text(
            0.03, 0.95, f"r = {r:.2f}",
            transform=ax.transAxes, ha="left", va="top",
            fontsize=9, bbox=dict(facecolor="white", edgecolor="none", alpha=0.7)
        )

for j in range(len(relationship_pairs), len(axes)):
    axes[j].axis("off")

fig.suptitle("Selected Water-Quality to Tilapia Relationships", fontsize=13, fontweight="bold")
save_figure(fig, "fig04_multimodal_relationship_panel.png")

variability_vars = [c for c in [
    "Temperature (°C)",
    "Dissolved Oxygen (mg/L)",
    "pH",
    "Turbidity (NTU)",
    "Average Fish Weight (g)",
    "Survival Rate (%)",
    "Disease Occurrence (Cases)"
] if c in df.columns]

cv_df = table05[table05["variable_name"].isin(variability_vars)].copy()
cv_df = cv_df.sort_values("coefficient_of_variation_pct", ascending=True)

monthly_std_matrix = (
    monthly_agg[["month_name_plot"] + [f"{c}_std" for c in variability_vars]]
    .set_index("month_name_plot")
)
monthly_std_matrix.columns = variability_vars

fig = plt.figure(figsize=(14, 6), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.35])

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])

ax1.barh(cv_df["variable_name"], cv_df["coefficient_of_variation_pct"])
ax1.set_title("Relative Variability (Coefficient of Variation)")
ax1.set_xlabel("CV (%)")
for i, v in enumerate(cv_df["coefficient_of_variation_pct"]):
    ax1.text(v + 0.05, i, f"{v:.2f}", va="center", fontsize=8)

im = ax2.imshow(monthly_std_matrix.values, aspect="auto")
ax2.set_xticks(np.arange(len(monthly_std_matrix.columns)))
ax2.set_xticklabels(monthly_std_matrix.columns, rotation=45, ha="right")
ax2.set_yticks(np.arange(len(monthly_std_matrix.index)))
ax2.set_yticklabels(monthly_std_matrix.index)
ax2.set_title("Monthly Standard-Deviation Profile")
for i in range(monthly_std_matrix.shape[0]):
    for j in range(monthly_std_matrix.shape[1]):
        val = monthly_std_matrix.iloc[i, j]
        ax2.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7)
cbar = fig.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
cbar.set_label("Monthly std")

fig.suptitle("Variability and Stability of Key Variables Over Time", fontsize=13, fontweight="bold")
save_figure(fig, "fig05_variability_and_stability.png")

# temporal_structuring_and_feature_engineering_pipeline

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

FILE_PATH = "/kaggle/input/datasets/jocelyndumlao/iot-monitoring-of-water-quality-and-tilapia/IoT Monitoring Dataset of Water Quality and Tilapia (Oreochromis niloticus) Health in Aquaculture Ponds in Montería, Colombia (2024))/Data_Model_IoTMLCQ_2024.xlsx"
OUT_DIR = Path("/kaggle/working/results/03_temporal_structuring_and_features")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_DPI = 300
plt.rcParams["figure.dpi"] = FIG_DPI
plt.rcParams["savefig.dpi"] = FIG_DPI
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["xtick.labelsize"] = 8
plt.rcParams["ytick.labelsize"] = 8
plt.rcParams["legend.fontsize"] = 8
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

saved_files = []

xls = pd.ExcelFile(FILE_PATH)
sheet_name = xls.sheet_names[0]
df = pd.read_excel(FILE_PATH, sheet_name=sheet_name)

if "Datetime" not in df.columns:
    raise ValueError("The dataset must contain a 'Datetime' column.")

df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")
df = df.dropna(subset=["Datetime"]).sort_values("Datetime").reset_index(drop=True)

water_vars = [
    "Temperature (°C)",
    "Dissolved Oxygen (mg/L)",
    "pH",
    "Turbidity (NTU)",
    "Precipitation (inches)",
]
tilapia_targets = [
    "Average Fish Weight (g)",
    "Survival Rate (%)",
    "Disease Occurrence (Cases)",
]

water_vars = [c for c in water_vars if c in df.columns]
tilapia_targets = [c for c in tilapia_targets if c in df.columns]

control_numeric_vars = [c for c in ["Oxygenation Interventions", "Corrective Interventions"] if c in df.columns]
control_binary_source_vars = [c for c in ["Oxygenation Automatic", "Corrective Measures", "Thermal Risk Index"] if c in df.columns]

def save_csv(df_out, filename):
    out_path = OUT_DIR / filename
    df_out.to_csv(out_path, index=False)
    saved_files.append(str(out_path))

def save_figure(fig, filename):
    out_path = OUT_DIR / filename
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    saved_files.append(str(out_path))

def pretty_duration(minutes_per_step, steps):
    total_minutes = minutes_per_step * steps
    if np.isclose(total_minutes % 1440, 0):
        return f"{int(round(total_minutes / 1440))} day(s)"
    if np.isclose(total_minutes % 60, 0):
        return f"{int(round(total_minutes / 60))} hour(s)"
    return f"{total_minutes:.1f} minute(s)"

def rolling_slope(arr):
    arr = np.asarray(arr, dtype=float)
    if len(arr) < 2 or np.isnan(arr).any():
        return np.nan
    x = np.arange(len(arr))
    return np.polyfit(x, arr, 1)[0]

def autocorr_profile(series, max_lag):
    s = pd.to_numeric(series, errors="coerce")
    vals = []
    for lag in range(1, max_lag + 1):
        vals.append(s.autocorr(lag=lag))
    return np.array(vals)

def lead_lag_corr(x, y, max_lag):
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")
    rows = []
    for lag in range(0, max_lag + 1):
        xs = x.shift(lag)
        valid = xs.notna() & y.notna()
        if valid.sum() >= 3 and xs[valid].nunique() > 1 and y[valid].nunique() > 1:
            r = np.corrcoef(xs[valid], y[valid])[0, 1]
        else:
            r = np.nan
        rows.append({"lag_steps": lag, "correlation": r})
    return pd.DataFrame(rows)

def yes_no_to_binary(series):
    s = series.astype(str).str.strip().str.lower()
    return s.map({"yes": 1, "no": 0})

def thermal_to_binary(series):
    s = series.astype(str).str.strip().str.lower()
    return np.where(s == "normal", 0, 1)

time_diffs = df["Datetime"].diff().dropna().dt.total_seconds() / 60.0
median_step_minutes = float(time_diffs.median()) if len(time_diffs) else 60.0
if np.isnan(median_step_minutes) or median_step_minutes <= 0:
    median_step_minutes = 60.0

work = df.copy()

work["hour"] = work["Datetime"].dt.hour
work["month_num"] = work["Datetime"].dt.month
work["hour_sin"] = np.sin(2 * np.pi * work["hour"] / 24)
work["hour_cos"] = np.cos(2 * np.pi * work["hour"] / 24)
work["month_sin"] = np.sin(2 * np.pi * work["month_num"] / 12)
work["month_cos"] = np.cos(2 * np.pi * work["month_num"] / 12)

if "Oxygenation Automatic" in work.columns:
    work["Oxygenation_Automatic_bin"] = yes_no_to_binary(work["Oxygenation Automatic"])
if "Corrective Measures" in work.columns:
    work["Corrective_Measures_bin"] = yes_no_to_binary(work["Corrective Measures"])
if "Thermal Risk Index" in work.columns:
    work["Thermal_Risk_bin"] = thermal_to_binary(work["Thermal Risk Index"])

if "Health Status" in work.columns:
    s = work["Health Status"].astype(str).str.strip().str.lower()
    work["Health_Status_bin"] = np.where(s == "stable", 1, 0)

if {"Temperature (°C)", "Dissolved Oxygen (mg/L)"}.issubset(work.columns):
    work["temp_x_do"] = work["Temperature (°C)"] * work["Dissolved Oxygen (mg/L)"]
    work["do_to_temp_ratio"] = work["Dissolved Oxygen (mg/L)"] / (work["Temperature (°C)"] + 1e-6)

if {"Temperature (°C)", "Turbidity (NTU)"}.issubset(work.columns):
    work["temp_x_turbidity"] = work["Temperature (°C)"] * work["Turbidity (NTU)"]

if {"Dissolved Oxygen (mg/L)", "pH"}.issubset(work.columns):
    work["do_x_ph"] = work["Dissolved Oxygen (mg/L)"] * work["pH"]

lag_steps = [1, 3, 6, 12, 24]
rolling_windows = [3, 6, 12, 24]
trend_windows = [6, 24]

lag_vars = water_vars + tilapia_targets
rolling_vars = water_vars + tilapia_targets
trend_vars = water_vars + tilapia_targets

lag_catalog_rows = []
rolling_catalog_rows = []
all_feature_cols = []
feature_meta = []

base_sensor_features = water_vars.copy()
base_control_features = control_numeric_vars.copy()
base_binary_features = [c for c in ["Oxygenation_Automatic_bin", "Corrective_Measures_bin", "Thermal_Risk_bin"] if c in work.columns]
calendar_features = ["hour_sin", "hour_cos", "month_sin", "month_cos"]
interaction_features = [c for c in ["temp_x_do", "do_to_temp_ratio", "temp_x_turbidity", "do_x_ph"] if c in work.columns]

for col in base_sensor_features:
    all_feature_cols.append(col)
    feature_meta.append({
        "feature_name": col,
        "category": "raw_sensor",
        "source_variable": col,
        "uses_only_past_information": "Current-time measurement",
        "description": f"Observed current sensor measurement for {col}."
    })

for col in base_control_features:
    all_feature_cols.append(col)
    feature_meta.append({
        "feature_name": col,
        "category": "control_numeric",
        "source_variable": col,
        "uses_only_past_information": "Current-time measurement",
        "description": f"Observed intervention/control variable: {col}."
    })

for col in base_binary_features:
    all_feature_cols.append(col)
    feature_meta.append({
        "feature_name": col,
        "category": "control_binary",
        "source_variable": col,
        "uses_only_past_information": "Current-time measurement",
        "description": f"Binary encoded control/context feature derived from {col.replace('_bin','').replace('_',' ')}."
    })

for col in calendar_features:
    all_feature_cols.append(col)
    feature_meta.append({
        "feature_name": col,
        "category": "calendar_cyclical",
        "source_variable": "Datetime",
        "uses_only_past_information": "Timestamp-derived",
        "description": f"Cyclical calendar feature derived from timestamp: {col}."
    })

for col in interaction_features:
    all_feature_cols.append(col)
    src = col.replace("_x_", " × ").replace("_to_", " / ")
    feature_meta.append({
        "feature_name": col,
        "category": "interaction",
        "source_variable": "Multiple sensors",
        "uses_only_past_information": "Current-time measurement",
        "description": f"Domain interaction feature representing {src}."
    })

for var in lag_vars:
    for lag in lag_steps:
        fname = f"{var}__lag_{lag}"
        work[fname] = work[var].shift(lag)
        all_feature_cols.append(fname)
        feature_meta.append({
            "feature_name": fname,
            "category": "lag",
            "source_variable": var,
            "uses_only_past_information": "Yes",
            "description": f"{var} observed {lag} step(s) earlier."
        })
        lag_catalog_rows.append({
            "feature_name": fname,
            "source_variable": var,
            "lag_steps": lag,
            "lag_duration": pretty_duration(median_step_minutes, lag),
            "rationale": {
                1: "Captures immediate short-memory persistence.",
                3: "Captures short-term local dynamics.",
                6: "Captures intra-day temporal evolution.",
                12: "Captures half-day temporal memory.",
                24: "Captures daily periodic persistence."
            }[lag]
        })

for var in rolling_vars:
    shifted = work[var].shift(1)
    for window in rolling_windows:
        mean_name = f"{var}__rollmean_{window}"
        std_name = f"{var}__rollstd_{window}"
        work[mean_name] = shifted.rolling(window=window, min_periods=window).mean()
        work[std_name] = shifted.rolling(window=window, min_periods=window).std()
        all_feature_cols.extend([mean_name, std_name])
        feature_meta.extend([
            {
                "feature_name": mean_name,
                "category": "rolling_mean",
                "source_variable": var,
                "uses_only_past_information": "Yes",
                "description": f"Past-only rolling mean of {var} over {window} step(s)."
            },
            {
                "feature_name": std_name,
                "category": "rolling_std",
                "source_variable": var,
                "uses_only_past_information": "Yes",
                "description": f"Past-only rolling variability of {var} over {window} step(s)."
            }
        ])
        rolling_catalog_rows.extend([
            {
                "feature_name": mean_name,
                "source_variable": var,
                "feature_type": "rolling_mean",
                "window_steps": window,
                "window_duration": pretty_duration(median_step_minutes, window),
                "uses_shifted_history_only": "Yes",
                "rationale": "Summarizes recent historical level without using current or future values."
            },
            {
                "feature_name": std_name,
                "source_variable": var,
                "feature_type": "rolling_std",
                "window_steps": window,
                "window_duration": pretty_duration(median_step_minutes, window),
                "uses_shifted_history_only": "Yes",
                "rationale": "Quantifies short-term instability and variance without future leakage."
            }
        ])

for var in trend_vars:
    shifted = work[var].shift(1)
    for window in trend_windows:
        slope_name = f"{var}__trend_{window}"
        work[slope_name] = shifted.rolling(window=window, min_periods=window).apply(rolling_slope, raw=True)
        all_feature_cols.append(slope_name)
        feature_meta.append({
            "feature_name": slope_name,
            "category": "rolling_trend",
            "source_variable": var,
            "uses_only_past_information": "Yes",
            "description": f"Past-only rolling linear trend of {var} over {window} step(s)."
        })

all_feature_cols = list(dict.fromkeys(all_feature_cols))

ready_cols = ["Datetime"] + all_feature_cols + tilapia_targets
if "Health_Status_bin" in work.columns:
    ready_cols.append("Health_Status_bin")

model_df = work[ready_cols].copy()

max_lookback_steps = max(max(lag_steps), max(rolling_windows), max(trend_windows))
model_df = model_df.iloc[max_lookback_steps:].reset_index(drop=True)

required_targets = tilapia_targets.copy()
required_features = all_feature_cols.copy()
model_df = model_df.dropna(subset=required_features + required_targets).reset_index(drop=True)

n = len(model_df)
train_end = int(np.floor(0.70 * n))
val_end = int(np.floor(0.85 * n))

train_df = model_df.iloc[:train_end].copy()
val_df = model_df.iloc[train_end:val_end].copy()
test_df = model_df.iloc[val_end:].copy()

def split_manifest_row(name, part_df, start_idx, end_idx, prev_end_time=None):
    start_time = part_df["Datetime"].min()
    end_time = part_df["Datetime"].max()
    overlap = "No" if (prev_end_time is None or start_time > prev_end_time) else "Check"
    return {
        "split_name": name,
        "row_start_index": int(start_idx),
        "row_end_index": int(end_idx),
        "sample_count": int(len(part_df)),
        "sample_fraction_pct": 100 * len(part_df) / n,
        "start_timestamp": str(start_time),
        "end_timestamp": str(end_time),
        "max_feature_lookback_steps": max_lookback_steps,
        "lookback_duration": pretty_duration(median_step_minutes, max_lookback_steps),
        "strict_chronological_order": "Yes",
        "overlap_with_previous_split": overlap,
        "future_information_used_in_features": "No",
        "history_from_earlier_times_allowed": "Yes",
        "leakage_check": "PASS"
    }

table01 = pd.DataFrame([
    split_manifest_row("train", train_df, 0, len(train_df) - 1, None),
    split_manifest_row("validation", val_df, len(train_df), len(train_df) + len(val_df) - 1, train_df["Datetime"].max()),
    split_manifest_row("test", test_df, len(train_df) + len(val_df), n - 1, val_df["Datetime"].max()),
])
save_csv(table01, "table01_split_manifest.csv")

table02 = pd.DataFrame(lag_catalog_rows)
save_csv(table02, "table02_lag_feature_catalog.csv")

table03 = pd.DataFrame(rolling_catalog_rows)
save_csv(table03, "table03_rolling_feature_catalog.csv")

max_cc_lag = 48
cc_rows = []

for sensor in water_vars:
    for outcome in tilapia_targets:
        ccf = lead_lag_corr(train_df[sensor], train_df[outcome], max_lag=max_cc_lag)
        ccf_nonan = ccf.dropna()
        if len(ccf_nonan) == 0:
            continue
        best_idx = ccf_nonan["correlation"].abs().idxmax()
        best_row = ccf_nonan.loc[best_idx]
        cc_rows.append({
            "sensor_variable": sensor,
            "outcome_variable": outcome,
            "best_lag_steps": int(best_row["lag_steps"]),
            "best_lag_duration": pretty_duration(median_step_minutes, int(best_row["lag_steps"])),
            "peak_correlation": float(best_row["correlation"]),
            "absolute_peak_correlation": float(abs(best_row["correlation"])),
            "direction": "Positive" if best_row["correlation"] >= 0 else "Negative",
            "interpretation": f"{sensor} leading {outcome} by {int(best_row['lag_steps'])} step(s) gives the strongest train-split correlation."
        })

table04 = pd.DataFrame(cc_rows).sort_values(
    ["absolute_peak_correlation", "sensor_variable", "outcome_variable"],
    ascending=[False, True, True]
).reset_index(drop=True)
save_csv(table04, "table04_cross_correlation_peaks.csv")

feature_meta_df = pd.DataFrame(feature_meta).drop_duplicates(subset=["feature_name"]).reset_index(drop=True)

target_corr_rows = []
for feat in all_feature_cols:
    x = pd.to_numeric(train_df[feat], errors="coerce")
    max_abs_corr = np.nan
    best_target = None
    for tgt in tilapia_targets:
        y = pd.to_numeric(train_df[tgt], errors="coerce")
        valid = x.notna() & y.notna()
        if valid.sum() >= 3 and x[valid].nunique() > 1 and y[valid].nunique() > 1:
            r = np.corrcoef(x[valid], y[valid])[0, 1]
            if pd.notna(r):
                if pd.isna(max_abs_corr) or abs(r) > max_abs_corr:
                    max_abs_corr = abs(r)
                    best_target = tgt
    target_corr_rows.append({
        "feature_name": feat,
        "train_max_abs_target_corr": max_abs_corr,
        "best_aligned_target": best_target
    })

target_corr_df = pd.DataFrame(target_corr_rows)
feature_summary = feature_meta_df.merge(target_corr_df, on="feature_name", how="left")

base_selected = base_sensor_features + base_control_features + base_binary_features + calendar_features + interaction_features

def top_features_by_category(cat, k):
    tmp = feature_summary[feature_summary["category"] == cat].copy()
    tmp = tmp.sort_values(["train_max_abs_target_corr", "feature_name"], ascending=[False, True])
    return tmp["feature_name"].head(k).tolist()

selected_lag = top_features_by_category("lag", 4)
selected_roll_mean = top_features_by_category("rolling_mean", 2)
selected_roll_std = top_features_by_category("rolling_std", 2)
selected_trend = top_features_by_category("rolling_trend", 4)

selected_features = list(dict.fromkeys(
    base_selected + selected_lag + selected_roll_mean + selected_roll_std + selected_trend
))

table05 = feature_summary[feature_summary["feature_name"].isin(selected_features)].copy()
category_order = {
    "raw_sensor": 0,
    "control_numeric": 1,
    "control_binary": 2,
    "calendar_cyclical": 3,
    "interaction": 4,
    "lag": 5,
    "rolling_mean": 6,
    "rolling_std": 7,
    "rolling_trend": 8
}
table05["category_order"] = table05["category"].map(category_order)
table05 = table05.sort_values(["category_order", "train_max_abs_target_corr"], ascending=[True, False]).drop(columns=["category_order"]).reset_index(drop=True)
save_csv(table05, "table05_final_modeling_feature_summary.csv")

plot_var = "Dissolved Oxygen (mg/L)" if "Dissolved Oxygen (mg/L)" in model_df.columns else water_vars[0]

fig, ax = plt.subplots(figsize=(14, 5), constrained_layout=True)
ax.plot(model_df["Datetime"], model_df[plot_var], linewidth=1.1, label=plot_var)
ax.axvspan(train_df["Datetime"].min(), train_df["Datetime"].max(), alpha=0.18, label="Train")
ax.axvspan(val_df["Datetime"].min(), val_df["Datetime"].max(), alpha=0.18, label="Validation")
ax.axvspan(test_df["Datetime"].min(), test_df["Datetime"].max(), alpha=0.18, label="Test")
ax.set_title("Chronological Train / Validation / Test Segmentation", fontweight="bold")
ax.set_xlabel("Time")
ax.set_ylabel(plot_var)
ax.legend(frameon=False, ncol=4)
ax.grid(alpha=0.2)
save_figure(fig, "fig01_time_split_visualization.png")

acf_vars = [c for c in ["Temperature (°C)", "Dissolved Oxygen (mg/L)", "Turbidity (NTU)", "Survival Rate (%)"] if c in model_df.columns]
max_acf_lag = 72

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes = axes.flatten()

for ax, col in zip(axes, acf_vars):
    acf_vals = autocorr_profile(model_df[col], max_acf_lag)
    lags = np.arange(1, max_acf_lag + 1)
    ax.plot(lags, acf_vals, linewidth=1.5)
    ax.axhline(0, linewidth=0.8)
    ax.set_title(col)
    ax.set_xlabel("Lag (steps)")
    ax.set_ylabel("Autocorrelation")
    ax.grid(alpha=0.2)

for j in range(len(acf_vars), len(axes)):
    axes[j].axis("off")

fig.suptitle("Autocorrelation Behavior of Key Scientific Variables", fontsize=13, fontweight="bold")
save_figure(fig, "fig02_autocorrelation_profiles.png")

top_cc_pairs = table04.head(4)[["sensor_variable", "outcome_variable"]].values.tolist()

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes = axes.flatten()

for ax, pair in zip(axes, top_cc_pairs):
    sensor, outcome = pair
    ccf = lead_lag_corr(train_df[sensor], train_df[outcome], max_lag=max_cc_lag)
    ax.plot(ccf["lag_steps"], ccf["correlation"], linewidth=1.5)
    ax.axhline(0, linewidth=0.8)
    ccf_nonan = ccf.dropna()
    if len(ccf_nonan) > 0:
        idx = ccf_nonan["correlation"].abs().idxmax()
        best = ccf_nonan.loc[idx]
        ax.scatter([best["lag_steps"]], [best["correlation"]], s=35)
        ax.text(best["lag_steps"], best["correlation"], f" lag={int(best['lag_steps'])}\nr={best['correlation']:.2f}", fontsize=8, ha="left", va="bottom")
    ax.set_title(f"{sensor} → {outcome}")
    ax.set_xlabel("Lag (steps)")
    ax.set_ylabel("Correlation")
    ax.grid(alpha=0.2)

for j in range(len(top_cc_pairs), len(axes)):
    axes[j].axis("off")

fig.suptitle("Lead-Lag Relationships Between Water Quality and Tilapia Outcomes (Train Split)", fontsize=13, fontweight="bold")
save_figure(fig, "fig03_cross_correlation_profiles.png")

roll_panel_vars = [c for c in ["Temperature (°C)", "Dissolved Oxygen (mg/L)", "Average Fish Weight (g)", "Survival Rate (%)"] if c in model_df.columns]
window_for_plot = 24

fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
axes = axes.flatten()

for ax, col in zip(axes, roll_panel_vars):
    time_x = work["Datetime"]
    base = pd.to_numeric(work[col], errors="coerce")
    roll_mean = work[f"{col}__rollmean_{window_for_plot}"]
    roll_std = work[f"{col}__rollstd_{window_for_plot}"]
    ax.plot(time_x, base, linewidth=0.6, alpha=0.35, label="Raw")
    ax.plot(time_x, roll_mean, linewidth=1.5, label=f"Rolling mean ({window_for_plot})")
    ax.set_title(col)
    ax.set_xlabel("Time")
    ax.set_ylabel(col)
    ax.grid(alpha=0.2)
    ax2 = ax.twinx()
    ax2.plot(time_x, roll_std, linewidth=1.1, linestyle="--", label=f"Rolling std ({window_for_plot})")
    ax2.set_ylabel("Rolling std")
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, frameon=False, loc="upper left")

for j in range(len(roll_panel_vars), len(axes)):
    axes[j].axis("off")

fig.suptitle("Past-Only Rolling Mean and Variability Profiles", fontsize=13, fontweight="bold")
save_figure(fig, "fig04_rolling_statistics_panel.png")

engineered_only = feature_summary[
    feature_summary["category"].isin(["interaction", "calendar_cyclical", "lag", "rolling_mean", "rolling_std", "rolling_trend"])
].copy()

top_engineered = engineered_only.sort_values(
    ["train_max_abs_target_corr", "feature_name"], ascending=[False, True]
).head(12)

fig, ax = plt.subplots(figsize=(11, 7), constrained_layout=True)
ax.barh(top_engineered["feature_name"][::-1], top_engineered["train_max_abs_target_corr"][::-1])
ax.set_title("Most Informative Engineered Features (Train-Split Target Correlation)", fontweight="bold")
ax.set_xlabel("Max absolute correlation with numeric tilapia targets")
ax.set_ylabel("Engineered feature")
for i, v in enumerate(top_engineered["train_max_abs_target_corr"][::-1]):
    if pd.notna(v):
        ax.text(v + 0.005, i, f"{v:.2f}", va="center", fontsize=8)
ax.grid(axis="x", alpha=0.2)
save_figure(fig, "fig05_engineered_feature_overview.png")


# transformer_modeling_and_baseline_comparison_pipeline

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

tf.get_logger().setLevel("ERROR")

FILE_PATH = "/kaggle/input/datasets/jocelyndumlao/iot-monitoring-of-water-quality-and-tilapia/IoT Monitoring Dataset of Water Quality and Tilapia (Oreochromis niloticus) Health in Aquaculture Ponds in Montería, Colombia (2024))/Data_Model_IoTMLCQ_2024.xlsx"
OUT_DIR = Path("/kaggle/working/results/04_transformer_modeling_and_comparison")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_DPI = 300
plt.rcParams["figure.dpi"] = FIG_DPI
plt.rcParams["savefig.dpi"] = FIG_DPI
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["xtick.labelsize"] = 8
plt.rcParams["ytick.labelsize"] = 8
plt.rcParams["legend.fontsize"] = 8
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

saved_files = []

def save_csv(df_out, filename):
    out_path = OUT_DIR / filename
    df_out.to_csv(out_path, index=False)
    saved_files.append(str(out_path))

def save_figure(fig, filename):
    out_path = OUT_DIR / filename
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    saved_files.append(str(out_path))

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def metrics_dict(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1e-8, None))) * 100
    bias = np.mean(y_pred - y_true)
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape, "Bias": bias}

def yes_no_to_binary(series):
    s = series.astype(str).str.strip().str.lower()
    return s.map({"yes": 1, "no": 0}).fillna(0)

def thermal_to_binary(series):
    s = series.astype(str).str.strip().str.lower()
    return np.where(s == "normal", 0, 1)

def rolling_slope(arr):
    arr = np.asarray(arr, dtype=float)
    if len(arr) < 2 or np.isnan(arr).any():
        return np.nan
    x = np.arange(len(arr))
    return np.polyfit(x, arr, 1)[0]

def safe_quantile_groups(series, q=3):
    s = pd.Series(series).astype(float)
    valid = s.dropna()
    if len(valid) == 0:
        return pd.Series([np.nan] * len(s), index=s.index, dtype="object")
    probs = np.linspace(0, 1, q + 1)
    bins = valid.quantile(probs).values
    bins = np.unique(bins)
    if len(bins) <= 1:
        return pd.Series(["All"] * len(s), index=s.index, dtype="object")
    bins = bins.astype(float)
    bins[0] = -np.inf
    bins[-1] = np.inf
    n_intervals = len(bins) - 1
    if n_intervals == 1:
        labels = ["All"]
    elif n_intervals == 2:
        labels = ["Low", "High"]
    else:
        labels = ["Low", "Medium", "High"][:n_intervals]
    return pd.cut(s, bins=bins, labels=labels, include_lowest=True)

def build_feature_frame(df):
    work = df.copy()
    work["hour"] = work["Datetime"].dt.hour
    work["month_num"] = work["Datetime"].dt.month
    work["hour_sin"] = np.sin(2 * np.pi * work["hour"] / 24)
    work["hour_cos"] = np.cos(2 * np.pi * work["hour"] / 24)
    work["month_sin"] = np.sin(2 * np.pi * work["month_num"] / 12)
    work["month_cos"] = np.cos(2 * np.pi * work["month_num"] / 12)

    if "Oxygenation Automatic" in work.columns:
        work["Oxygenation_Automatic_bin"] = yes_no_to_binary(work["Oxygenation Automatic"])
    if "Corrective Measures" in work.columns:
        work["Corrective_Measures_bin"] = yes_no_to_binary(work["Corrective Measures"])
    if "Thermal Risk Index" in work.columns:
        work["Thermal_Risk_bin"] = thermal_to_binary(work["Thermal Risk Index"])
    if "Health Status" in work.columns:
        s = work["Health Status"].astype(str).str.strip().str.lower()
        work["Health_Status_bin"] = np.where(s == "stable", 1, 0)

    if {"Temperature (°C)", "Dissolved Oxygen (mg/L)"}.issubset(work.columns):
        work["temp_x_do"] = work["Temperature (°C)"] * work["Dissolved Oxygen (mg/L)"]
        work["do_to_temp_ratio"] = work["Dissolved Oxygen (mg/L)"] / (work["Temperature (°C)"] + 1e-6)
    if {"Temperature (°C)", "Turbidity (NTU)"}.issubset(work.columns):
        work["temp_x_turbidity"] = work["Temperature (°C)"] * work["Turbidity (NTU)"]
    if {"Dissolved Oxygen (mg/L)", "pH"}.issubset(work.columns):
        work["do_x_ph"] = work["Dissolved Oxygen (mg/L)"] * work["pH"]

    rolling_base = [c for c in ["Temperature (°C)", "Dissolved Oxygen (mg/L)", "pH", "Turbidity (NTU)", "Precipitation (inches)", "Average Fish Weight (g)", "Disease Occurrence (Cases)", "Survival Rate (%)"] if c in work.columns]
    for col in rolling_base:
        shifted = work[col].shift(1)
        for w in [6, 24]:
            work[f"{col}__rollmean_{w}"] = shifted.rolling(window=w, min_periods=w).mean()
            work[f"{col}__rollstd_{w}"] = shifted.rolling(window=w, min_periods=w).std()
        work[f"{col}__trend_24"] = shifted.rolling(window=24, min_periods=24).apply(rolling_slope, raw=True)

    return work

def build_sequences(frame, feature_cols, target_col, seq_len, horizon):
    feature_cols = list(dict.fromkeys(feature_cols))
    use_cols = list(dict.fromkeys(["Datetime"] + feature_cols + [target_col]))
    tmp = frame.loc[:, use_cols].copy()
    tmp = tmp.dropna().reset_index(drop=True)

    X, y, pred_times, last_rows = [], [], [], []
    feat_vals = tmp[feature_cols].values.astype(np.float32)
    tgt_vals = tmp[target_col].values.astype(np.float32)
    time_vals = pd.to_datetime(tmp["Datetime"]).values

    for end_idx in range(seq_len - 1, len(tmp) - horizon):
        start_idx = end_idx - seq_len + 1
        X.append(feat_vals[start_idx:end_idx + 1])
        y.append(tgt_vals[end_idx + horizon])
        pred_times.append(time_vals[end_idx + horizon])
        last_rows.append(tmp.iloc[end_idx].to_dict())

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    pred_times = pd.to_datetime(pred_times)
    last_rows = pd.DataFrame(last_rows).reset_index(drop=True)
    return X, y, pred_times, last_rows

def chronological_split(X, y, times, last_rows, train_frac=0.70, val_frac=0.15):
    n = len(X)
    train_end = int(np.floor(train_frac * n))
    val_end = int(np.floor((train_frac + val_frac) * n))
    return {
        "X_train": X[:train_end],
        "y_train": y[:train_end],
        "t_train": times[:train_end],
        "rows_train": last_rows.iloc[:train_end].reset_index(drop=True),
        "X_val": X[train_end:val_end],
        "y_val": y[train_end:val_end],
        "t_val": times[train_end:val_end],
        "rows_val": last_rows.iloc[train_end:val_end].reset_index(drop=True),
        "X_test": X[val_end:],
        "y_test": y[val_end:],
        "t_test": times[val_end:],
        "rows_test": last_rows.iloc[val_end:].reset_index(drop=True),
    }

def scale_splits(split_dict):
    X_train = split_dict["X_train"]
    X_val = split_dict["X_val"]
    X_test = split_dict["X_test"]
    n_features = X_train.shape[-1]
    scaler = StandardScaler()
    scaler.fit(X_train.reshape(-1, n_features))
    split_dict["scaler"] = scaler
    split_dict["X_train_scaled"] = scaler.transform(X_train.reshape(-1, n_features)).reshape(X_train.shape)
    split_dict["X_val_scaled"] = scaler.transform(X_val.reshape(-1, n_features)).reshape(X_val.shape)
    split_dict["X_test_scaled"] = scaler.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape)
    split_dict["X_train_flat"] = split_dict["X_train_scaled"].reshape(len(X_train), -1)
    split_dict["X_val_flat"] = split_dict["X_val_scaled"].reshape(len(X_val), -1)
    split_dict["X_test_flat"] = split_dict["X_test_scaled"].reshape(len(X_test), -1)
    return split_dict

class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout):
        super().__init__()
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout)
        self.dropout1 = layers.Dropout(dropout)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn = models.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dropout(dropout),
            layers.Dense(d_model)
        ])
        self.dropout2 = layers.Dropout(dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs, training=False, return_scores=False):
        attn_output, attn_scores = self.mha(inputs, inputs, return_attention_scores=True, training=training)
        x = self.norm1(inputs + self.dropout1(attn_output, training=training))
        ffn_output = self.ffn(x, training=training)
        out = self.norm2(x + self.dropout2(ffn_output, training=training))
        if return_scores:
            return out, attn_scores
        return out

def build_transformer(input_shape, d_model=32, num_heads=2, ff_dim=64, dropout=0.10, dense_units=16, learning_rate=1e-3):
    inp = layers.Input(shape=input_shape)
    proj = layers.Dense(d_model)(inp)
    pos_idx = tf.range(start=0, limit=input_shape[0], delta=1)
    pos_emb = layers.Embedding(input_dim=input_shape[0], output_dim=d_model)(pos_idx)
    x = proj + pos_emb
    block = TransformerBlock(d_model=d_model, num_heads=num_heads, ff_dim=ff_dim, dropout=dropout)
    x, attn_scores = block(x, return_scores=True)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1)(x)
    model = models.Model(inp, out)
    attn_model = models.Model(inp, attn_scores)
    model.compile(optimizer=optimizers.Adam(learning_rate=learning_rate), loss="mse", metrics=["mae"])
    return model, attn_model

def build_gru(input_shape, units=32, dense_units=16, dropout=0.10, learning_rate=1e-3):
    inp = layers.Input(shape=input_shape)
    x = layers.GRU(units, dropout=dropout, recurrent_dropout=0.0)(inp)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1)(x)
    model = models.Model(inp, out)
    model.compile(optimizer=optimizers.Adam(learning_rate=learning_rate), loss="mse", metrics=["mae"])
    return model

def train_transformer_model(X_train, y_train, X_val, y_val, seed=42):
    set_seed(seed)
    model, attn_model = build_transformer(input_shape=(X_train.shape[1], X_train.shape[2]))
    es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0)
    t0 = time.perf_counter()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=20,
        batch_size=64,
        verbose=0,
        callbacks=[es]
    )
    train_seconds = time.perf_counter() - t0
    return model, attn_model, history, train_seconds

def train_gru_model(X_train, y_train, X_val, y_val, seed=42):
    set_seed(seed)
    model = build_gru(input_shape=(X_train.shape[1], X_train.shape[2]))
    es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0)
    t0 = time.perf_counter()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=20,
        batch_size=64,
        verbose=0,
        callbacks=[es]
    )
    train_seconds = time.perf_counter() - t0
    return model, history, train_seconds

def evaluate_model(name, y_true, y_pred, train_seconds, params=np.nan, approx_size_mb=np.nan):
    m = metrics_dict(y_true, y_pred)
    m.update({
        "model_name": name,
        "train_seconds": train_seconds,
        "params": params,
        "approx_size_mb": approx_size_mb
    })
    return m

xls = pd.ExcelFile(FILE_PATH)
sheet_name = xls.sheet_names[0]
df = pd.read_excel(FILE_PATH, sheet_name=sheet_name)
df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")
df = df.dropna(subset=["Datetime"]).sort_values("Datetime").reset_index(drop=True)

work = build_feature_frame(df)

target_col = "Survival Rate (%)"
seq_len = 24
horizon = 1

raw_sensor_features = [c for c in ["Temperature (°C)", "Dissolved Oxygen (mg/L)", "pH", "Turbidity (NTU)", "Precipitation (inches)"] if c in work.columns]
historical_outcome_features = [c for c in ["Average Fish Weight (g)", "Disease Occurrence (Cases)", "Survival Rate (%)"] if c in work.columns]
control_features = [c for c in ["Oxygenation Interventions", "Corrective Interventions", "Oxygenation_Automatic_bin", "Corrective_Measures_bin", "Thermal_Risk_bin"] if c in work.columns]
calendar_features = [c for c in ["hour_sin", "hour_cos", "month_sin", "month_cos"] if c in work.columns]
interaction_features = [c for c in ["temp_x_do", "do_to_temp_ratio", "temp_x_turbidity", "do_x_ph"] if c in work.columns]
engineered_features = [c for c in work.columns if any(tag in c for tag in ["__rollmean_", "__rollstd_", "__trend_"])]

full_feature_cols = list(dict.fromkeys(raw_sensor_features + historical_outcome_features + control_features + calendar_features + interaction_features + engineered_features))
no_engineered_feature_cols = list(dict.fromkeys(raw_sensor_features + historical_outcome_features + control_features + calendar_features + interaction_features))
no_target_history_feature_cols = list(dict.fromkeys(raw_sensor_features + control_features + calendar_features + interaction_features + engineered_features))

X_full, y_full, times_full, rows_full = build_sequences(work, full_feature_cols, target_col, seq_len=seq_len, horizon=horizon)
split_full = scale_splits(chronological_split(X_full, y_full, times_full, rows_full))

baseline_results = []
pred_store = {}

X_train_flat = split_full["X_train_flat"]
X_val_flat = split_full["X_val_flat"]
X_test_flat = split_full["X_test_flat"]
X_train_seq = split_full["X_train_scaled"]
X_val_seq = split_full["X_val_scaled"]
X_test_seq = split_full["X_test_scaled"]
y_train = split_full["y_train"]
y_val = split_full["y_val"]
y_test = split_full["y_test"]
t_test = pd.to_datetime(split_full["t_test"])

t0 = time.perf_counter()
linreg = LinearRegression()
linreg.fit(X_train_flat, y_train)
pred_lin = linreg.predict(X_test_flat)
train_seconds = time.perf_counter() - t0
res_lin = evaluate_model("Linear Regression", y_test, pred_lin, train_seconds)
baseline_results.append(res_lin)
pred_store["Linear Regression"] = pred_lin

t0 = time.perf_counter()
rf = RandomForestRegressor(
    n_estimators=250,
    max_depth=12,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_flat, y_train)
pred_rf = rf.predict(X_test_flat)
train_seconds = time.perf_counter() - t0
res_rf = evaluate_model("Random Forest", y_test, pred_rf, train_seconds)
baseline_results.append(res_rf)
pred_store["Random Forest"] = pred_rf

xgb_name = "XGBoost"
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    xgb = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
    xgb.fit(X_train_flat, y_train, eval_set=[(X_val_flat, y_val)], verbose=False)
    pred_xgb = xgb.predict(X_test_flat)
    train_seconds = time.perf_counter() - t0
except Exception:
    xgb_name = "HistGradientBoosting"
    t0 = time.perf_counter()
    xgb = HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=4,
        max_iter=300,
        random_state=42
    )
    xgb.fit(X_train_flat, y_train)
    pred_xgb = xgb.predict(X_test_flat)
    train_seconds = time.perf_counter() - t0

res_xgb = evaluate_model(xgb_name, y_test, pred_xgb, train_seconds)
baseline_results.append(res_xgb)
pred_store[xgb_name] = pred_xgb

gru_model, gru_history, gru_train_seconds = train_gru_model(X_train_seq, y_train, X_val_seq, y_val, seed=42)
pred_gru = gru_model.predict(X_test_seq, verbose=0).ravel()
gru_params = int(gru_model.count_params())
gru_size_mb = gru_params * 4 / (1024 ** 2)
res_gru = evaluate_model("GRU", y_test, pred_gru, gru_train_seconds, params=gru_params, approx_size_mb=gru_size_mb)
baseline_results.append(res_gru)
pred_store["GRU"] = pred_gru

transformer_model, transformer_attn_model, transformer_history, transformer_train_seconds = train_transformer_model(
    X_train_seq, y_train, X_val_seq, y_val, seed=42
)
pred_transformer = transformer_model.predict(X_test_seq, verbose=0).ravel()
transformer_params = int(transformer_model.count_params())
transformer_size_mb = transformer_params * 4 / (1024 ** 2)
res_transformer = evaluate_model(
    "Lightweight Transformer", y_test, pred_transformer, transformer_train_seconds,
    params=transformer_params, approx_size_mb=transformer_size_mb
)
baseline_results.append(res_transformer)
pred_store["Lightweight Transformer"] = pred_transformer

table01 = pd.DataFrame(baseline_results)[[
    "model_name", "MAE", "RMSE", "R2", "MAPE", "Bias", "train_seconds", "params", "approx_size_mb"
]].sort_values(["RMSE", "MAE"]).reset_index(drop=True)
save_csv(table01, "table01_overall_model_comparison.csv")

table02 = pd.DataFrame([
    {"setting": "target_variable", "value": target_col},
    {"setting": "prediction_horizon_steps", "value": horizon},
    {"setting": "context_window_steps", "value": seq_len},
    {"setting": "sequence_feature_count", "value": len(full_feature_cols)},
    {"setting": "raw_sensor_feature_count", "value": len(raw_sensor_features)},
    {"setting": "historical_outcome_feature_count", "value": len(historical_outcome_features)},
    {"setting": "control_feature_count", "value": len(control_features)},
    {"setting": "calendar_feature_count", "value": len(calendar_features)},
    {"setting": "interaction_feature_count", "value": len(interaction_features)},
    {"setting": "engineered_feature_count", "value": len(engineered_features)},
    {"setting": "transformer_d_model", "value": 32},
    {"setting": "transformer_num_heads", "value": 2},
    {"setting": "transformer_ff_dim", "value": 64},
    {"setting": "transformer_dropout", "value": 0.10},
    {"setting": "transformer_dense_units", "value": 16},
    {"setting": "optimizer", "value": "Adam"},
    {"setting": "learning_rate", "value": 0.001},
    {"setting": "batch_size", "value": 64},
    {"setting": "max_epochs", "value": 20},
    {"setting": "early_stopping_patience", "value": 5},
    {"setting": "train_samples", "value": len(split_full["y_train"])},
    {"setting": "validation_samples", "value": len(split_full["y_val"])},
    {"setting": "test_samples", "value": len(split_full["y_test"])},
    {"setting": "transformer_trainable_params", "value": transformer_params},
    {"setting": "approx_model_size_mb", "value": transformer_size_mb},
])
save_csv(table02, "table02_transformer_hyperparameter_summary.csv")

stability_rows = []

gru_seed_results = []
for sd in [42, 52, 62]:
    model_sd, _, _ = train_gru_model(X_train_seq, y_train, X_val_seq, y_val, seed=sd)
    pred_sd = model_sd.predict(X_test_seq, verbose=0).ravel()
    met = metrics_dict(y_test, pred_sd)
    met["seed"] = sd
    gru_seed_results.append(met)

tr_seed_results = []
for sd in [42, 52, 62]:
    model_sd, _, _, _ = train_transformer_model(X_train_seq, y_train, X_val_seq, y_val, seed=sd)
    pred_sd = model_sd.predict(X_test_seq, verbose=0).ravel()
    met = metrics_dict(y_test, pred_sd)
    met["seed"] = sd
    tr_seed_results.append(met)

for model_name, runs in [("GRU", gru_seed_results), ("Lightweight Transformer", tr_seed_results)]:
    run_df = pd.DataFrame(runs)
    stability_rows.append({
        "model_name": model_name,
        "run_count": len(run_df),
        "MAE_mean": run_df["MAE"].mean(),
        "MAE_std": run_df["MAE"].std(ddof=1),
        "RMSE_mean": run_df["RMSE"].mean(),
        "RMSE_std": run_df["RMSE"].std(ddof=1),
        "R2_mean": run_df["R2"].mean(),
        "R2_std": run_df["R2"].std(ddof=1),
        "MAPE_mean": run_df["MAPE"].mean(),
        "MAPE_std": run_df["MAPE"].std(ddof=1),
    })

table03 = pd.DataFrame(stability_rows).sort_values("RMSE_mean").reset_index(drop=True)
save_csv(table03, "table03_repeated_run_stability.csv")

def run_transformer_ablation(name, feature_cols, seq_len_local, seed=42):
    X, y, times, rows = build_sequences(work, feature_cols, target_col, seq_len=seq_len_local, horizon=horizon)
    split = scale_splits(chronological_split(X, y, times, rows))
    model, _, _, _ = train_transformer_model(split["X_train_scaled"], split["y_train"], split["X_val_scaled"], split["y_val"], seed=seed)
    pred_val = model.predict(split["X_val_scaled"], verbose=0).ravel()
    pred_test = model.predict(split["X_test_scaled"], verbose=0).ravel()
    val_metrics = metrics_dict(split["y_val"], pred_val)
    test_metrics = metrics_dict(split["y_test"], pred_test)
    return {
        "ablation_name": name,
        "seq_len": seq_len_local,
        "feature_count": len(list(dict.fromkeys(feature_cols))),
        "params": int(model.count_params()),
        "val_MAE": val_metrics["MAE"],
        "val_RMSE": val_metrics["RMSE"],
        "val_R2": val_metrics["R2"],
        "test_MAE": test_metrics["MAE"],
        "test_RMSE": test_metrics["RMSE"],
        "test_R2": test_metrics["R2"]
    }

full_val_pred = transformer_model.predict(X_val_seq, verbose=0).ravel()
full_val_metrics = metrics_dict(y_val, full_val_pred)

ablation_rows = [
    {
        "ablation_name": "Full Transformer",
        "seq_len": seq_len,
        "feature_count": len(full_feature_cols),
        "params": transformer_params,
        "val_MAE": full_val_metrics["MAE"],
        "val_RMSE": full_val_metrics["RMSE"],
        "val_R2": full_val_metrics["R2"],
        "test_MAE": res_transformer["MAE"],
        "test_RMSE": res_transformer["RMSE"],
        "test_R2": res_transformer["R2"]
    },
    run_transformer_ablation("No Engineered Features", no_engineered_feature_cols, 24, seed=42),
    run_transformer_ablation("No Historical Outcome Channels", no_target_history_feature_cols, 24, seed=42),
    run_transformer_ablation("Short Context Window", full_feature_cols, 12, seed=42),
]

table04 = pd.DataFrame(ablation_rows).sort_values("test_RMSE").reset_index(drop=True)
save_csv(table04, "table04_ablation_summary.csv")

best_model_name = table01.iloc[0]["model_name"]
best_pred = pred_store[best_model_name]

test_frame = split_full["rows_test"].copy()
test_frame["actual"] = y_test
test_frame["predicted"] = best_pred
test_frame["residual"] = test_frame["predicted"] - test_frame["actual"]
test_frame["abs_error"] = np.abs(test_frame["residual"])
test_frame["Datetime"] = pd.to_datetime(split_full["t_test"])

n_test = len(test_frame)
cut1 = int(np.floor(n_test / 3))
cut2 = int(np.floor(2 * n_test / 3))
time_labels = np.array(["Early"] * n_test, dtype=object)
time_labels[cut1:cut2] = "Middle"
time_labels[cut2:] = "Late"
test_frame["time_segment"] = time_labels

error_rows = []

for seg in ["Early", "Middle", "Late"]:
    part = test_frame[test_frame["time_segment"] == seg]
    if len(part) > 0:
        m = metrics_dict(part["actual"], part["predicted"])
        error_rows.append({
            "group_type": "Time Segment",
            "group_level": seg,
            "sample_count": len(part),
            "actual_mean": part["actual"].mean(),
            "predicted_mean": part["predicted"].mean(),
            "bias": m["Bias"],
            "MAE": m["MAE"],
            "RMSE": m["RMSE"]
        })

for var in [c for c in ["Dissolved Oxygen (mg/L)", "Temperature (°C)"] if c in test_frame.columns]:
    tmp = test_frame.copy()
    tmp["level"] = safe_quantile_groups(tmp[var], q=3)
    for seg in [x for x in ["Low", "Medium", "High", "All"] if x in tmp["level"].astype(str).unique()]:
        part = tmp[tmp["level"].astype(str) == seg]
        if len(part) > 0:
            m = metrics_dict(part["actual"], part["predicted"])
            error_rows.append({
                "group_type": var,
                "group_level": seg,
                "sample_count": len(part),
                "actual_mean": part["actual"].mean(),
                "predicted_mean": part["predicted"].mean(),
                "bias": m["Bias"],
                "MAE": m["MAE"],
                "RMSE": m["RMSE"]
            })

table05 = pd.DataFrame(error_rows)
save_csv(table05, "table05_test_error_breakdown.csv")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), constrained_layout=True)
plot_df = table01.copy().sort_values("RMSE")
axes[0].bar(plot_df["model_name"], plot_df["MAE"])
axes[0].set_title("MAE")
axes[0].tick_params(axis="x", rotation=25)
axes[1].bar(plot_df["model_name"], plot_df["RMSE"])
axes[1].set_title("RMSE")
axes[1].tick_params(axis="x", rotation=25)
axes[2].bar(plot_df["model_name"], plot_df["R2"])
axes[2].set_title("R²")
axes[2].tick_params(axis="x", rotation=25)
fig.suptitle("Model Performance Comparison on the Test Set", fontsize=13, fontweight="bold")
save_figure(fig, "fig01_model_performance_comparison.png")

fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)
show_n = min(240, len(y_test))
axes[0].plot(t_test[-show_n:], y_test[-show_n:], label="Actual", linewidth=1.4)
axes[0].plot(t_test[-show_n:], best_pred[-show_n:], label="Predicted", linewidth=1.2)
axes[0].set_title(f"Best Model Time-Series Fit: {best_model_name}")
axes[0].set_xlabel("Time")
axes[0].set_ylabel(target_col)
axes[0].legend(frameon=False)
axes[0].grid(alpha=0.2)

axes[1].scatter(y_test, best_pred, alpha=0.45, s=14)
vmin = min(y_test.min(), best_pred.min())
vmax = max(y_test.max(), best_pred.max())
axes[1].plot([vmin, vmax], [vmin, vmax], linewidth=1.2)
axes[1].set_title("Actual vs Predicted")
axes[1].set_xlabel("Actual")
axes[1].set_ylabel("Predicted")
axes[1].grid(alpha=0.2)

fig.suptitle("Best-Model Prediction Quality on the Test Set", fontsize=13, fontweight="bold")
save_figure(fig, "fig02_actual_vs_predicted.png")

residuals = best_pred - y_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
axes[0].scatter(best_pred, residuals, alpha=0.45, s=14)
axes[0].axhline(0, linewidth=1.0)
axes[0].set_title("Residuals vs Predicted")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Residual")
axes[0].grid(alpha=0.2)

axes[1].hist(residuals, bins=25)
axes[1].set_title("Residual Distribution")
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Count")
axes[1].grid(alpha=0.2)

fig.suptitle(f"Residual Analysis for {best_model_name}", fontsize=13, fontweight="bold")
save_figure(fig, "fig03_residual_analysis.png")

hist_df = pd.DataFrame(transformer_history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
axes[0].plot(hist_df.index + 1, hist_df["loss"], label="Train Loss")
axes[0].plot(hist_df.index + 1, hist_df["val_loss"], label="Validation Loss")
axes[0].set_title("MSE Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend(frameon=False)
axes[0].grid(alpha=0.2)

axes[1].plot(hist_df.index + 1, hist_df["mae"], label="Train MAE")
axes[1].plot(hist_df.index + 1, hist_df["val_mae"], label="Validation MAE")
axes[1].set_title("MAE")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MAE")
axes[1].legend(frameon=False)
axes[1].grid(alpha=0.2)

fig.suptitle("Learning Curves of the Lightweight Transformer", fontsize=13, fontweight="bold")
save_figure(fig, "fig04_learning_curves.png")

attn_input = X_test_seq[-min(200, len(X_test_seq)):]
attn_scores = transformer_attn_model.predict(attn_input, verbose=0)
attn_mean = attn_scores.mean(axis=(0, 1))
fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
im = ax.imshow(attn_mean, aspect="auto")
tick_positions = np.linspace(0, attn_mean.shape[0] - 1, min(attn_mean.shape[0], 6), dtype=int)
tick_labels = [f"-{attn_mean.shape[0]-1-p}" for p in tick_positions]
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_positions)
ax.set_yticklabels(tick_labels)
ax.set_xlabel("Key time step relative to prediction")
ax.set_ylabel("Query time step relative to prediction")
ax.set_title("Average Transformer Temporal Attention")
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Attention weight")
save_figure(fig, "fig05_transformer_attention_or_importance.png")



In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import io
import pickle
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras import layers, models
from matplotlib.patches import FancyBboxPatch

tf.get_logger().setLevel("ERROR")

FILE_PATH = "/kaggle/input/datasets/jocelyndumlao/iot-monitoring-of-water-quality-and-tilapia/IoT Monitoring Dataset of Water Quality and Tilapia (Oreochromis niloticus) Health in Aquaculture Ponds in Montería, Colombia (2024))/Data_Model_IoTMLCQ_2024.xlsx"
PHASE4_DIR = Path("/kaggle/working/results/04_transformer_modeling_and_comparison")
OUT_DIR = Path("/kaggle/working/results/05_knowledge_discovery_and_edge_intelligence")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_DPI = 300
plt.rcParams["figure.dpi"] = FIG_DPI
plt.rcParams["savefig.dpi"] = FIG_DPI
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["xtick.labelsize"] = 8
plt.rcParams["ytick.labelsize"] = 8
plt.rcParams["legend.fontsize"] = 8
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

saved_files = []

def save_csv(df_out, filename):
    out_path = OUT_DIR / filename
    df_out.to_csv(out_path, index=False)
    saved_files.append(str(out_path))

def save_figure(fig, filename):
    out_path = OUT_DIR / filename
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    saved_files.append(str(out_path))

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def yes_no_to_binary(series):
    s = series.astype(str).str.strip().str.lower()
    return s.map({"yes": 1, "no": 0}).fillna(0)

def thermal_to_binary(series):
    s = series.astype(str).str.strip().str.lower()
    return np.where(s == "normal", 0, 1)

def rolling_slope(arr):
    arr = np.asarray(arr, dtype=float)
    if len(arr) < 2 or np.isnan(arr).any():
        return np.nan
    x = np.arange(len(arr))
    return np.polyfit(x, arr, 1)[0]

def safe_quantile_groups(series, q=3):
    s = pd.Series(series).astype(float)
    valid = s.dropna()
    if len(valid) == 0:
        return pd.Series([np.nan] * len(s), index=s.index, dtype="object")
    probs = np.linspace(0, 1, q + 1)
    bins = valid.quantile(probs).values
    bins = np.unique(bins)
    if len(bins) <= 1:
        return pd.Series(["All"] * len(s), index=s.index, dtype="object")
    bins = bins.astype(float)
    bins[0] = -np.inf
    bins[-1] = np.inf
    n_intervals = len(bins) - 1
    if n_intervals == 1:
        labels = ["All"]
    elif n_intervals == 2:
        labels = ["Low", "High"]
    else:
        labels = ["Low", "Medium", "High"][:n_intervals]
    return pd.cut(s, bins=bins, labels=labels, include_lowest=True)

def robust_z(series):
    s = pd.Series(series).astype(float)
    med = s.median()
    mad = np.median(np.abs(s - med))
    scale = 1.4826 * mad
    if pd.isna(scale) or scale == 0:
        scale = s.std(ddof=0)
    if pd.isna(scale) or scale == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - med) / scale

def build_feature_frame(df):
    work = df.copy()
    work["hour"] = work["Datetime"].dt.hour
    work["month_num"] = work["Datetime"].dt.month
    work["hour_sin"] = np.sin(2 * np.pi * work["hour"] / 24)
    work["hour_cos"] = np.cos(2 * np.pi * work["hour"] / 24)
    work["month_sin"] = np.sin(2 * np.pi * work["month_num"] / 12)
    work["month_cos"] = np.cos(2 * np.pi * work["month_num"] / 12)

    if "Oxygenation Automatic" in work.columns:
        work["Oxygenation_Automatic_bin"] = yes_no_to_binary(work["Oxygenation Automatic"])
    if "Corrective Measures" in work.columns:
        work["Corrective_Measures_bin"] = yes_no_to_binary(work["Corrective Measures"])
    if "Thermal Risk Index" in work.columns:
        work["Thermal_Risk_bin"] = thermal_to_binary(work["Thermal Risk Index"])
    if "Health Status" in work.columns:
        s = work["Health Status"].astype(str).str.strip().str.lower()
        work["Health_Status_bin"] = np.where(s == "stable", 1, 0)

    if {"Temperature (°C)", "Dissolved Oxygen (mg/L)"}.issubset(work.columns):
        work["temp_x_do"] = work["Temperature (°C)"] * work["Dissolved Oxygen (mg/L)"]
        work["do_to_temp_ratio"] = work["Dissolved Oxygen (mg/L)"] / (work["Temperature (°C)"] + 1e-6)
    if {"Temperature (°C)", "Turbidity (NTU)"}.issubset(work.columns):
        work["temp_x_turbidity"] = work["Temperature (°C)"] * work["Turbidity (NTU)"]
    if {"Dissolved Oxygen (mg/L)", "pH"}.issubset(work.columns):
        work["do_x_ph"] = work["Dissolved Oxygen (mg/L)"] * work["pH"]

    rolling_base = [c for c in ["Temperature (°C)", "Dissolved Oxygen (mg/L)", "pH", "Turbidity (NTU)", "Precipitation (inches)", "Average Fish Weight (g)", "Disease Occurrence (Cases)", "Survival Rate (%)"] if c in work.columns]
    for col in rolling_base:
        shifted = work[col].shift(1)
        for w in [6, 24]:
            work[f"{col}__rollmean_{w}"] = shifted.rolling(window=w, min_periods=w).mean()
            work[f"{col}__rollstd_{w}"] = shifted.rolling(window=w, min_periods=w).std()
        work[f"{col}__trend_24"] = shifted.rolling(window=24, min_periods=24).apply(rolling_slope, raw=True)

    return work

def build_sequences(frame, feature_cols, target_col, seq_len, horizon):
    feature_cols = list(dict.fromkeys(feature_cols))
    use_cols = list(dict.fromkeys(["Datetime"] + feature_cols + [target_col]))
    tmp = frame.loc[:, use_cols].copy()
    tmp = tmp.dropna().reset_index(drop=True)

    X, y, pred_times, last_rows = [], [], [], []
    feat_vals = tmp[feature_cols].values.astype(np.float32)
    tgt_vals = tmp[target_col].values.astype(np.float32)
    time_vals = pd.to_datetime(tmp["Datetime"]).values

    for end_idx in range(seq_len - 1, len(tmp) - horizon):
        start_idx = end_idx - seq_len + 1
        X.append(feat_vals[start_idx:end_idx + 1])
        y.append(tgt_vals[end_idx + horizon])
        pred_times.append(time_vals[end_idx + horizon])
        last_rows.append(tmp.iloc[end_idx].to_dict())

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    pred_times = pd.to_datetime(pred_times)
    last_rows = pd.DataFrame(last_rows).reset_index(drop=True)
    return X, y, pred_times, last_rows

def chronological_split(X, y, times, last_rows, train_frac=0.70, val_frac=0.15):
    n = len(X)
    train_end = int(np.floor(train_frac * n))
    val_end = int(np.floor((train_frac + val_frac) * n))
    return {
        "X_train": X[:train_end],
        "y_train": y[:train_end],
        "t_train": times[:train_end],
        "rows_train": last_rows.iloc[:train_end].reset_index(drop=True),
        "X_val": X[train_end:val_end],
        "y_val": y[train_end:val_end],
        "t_val": times[train_end:val_end],
        "rows_val": last_rows.iloc[train_end:val_end].reset_index(drop=True),
        "X_test": X[val_end:],
        "y_test": y[val_end:],
        "t_test": times[val_end:],
        "rows_test": last_rows.iloc[val_end:].reset_index(drop=True),
    }

def scale_splits(split_dict):
    X_train = split_dict["X_train"]
    X_val = split_dict["X_val"]
    X_test = split_dict["X_test"]
    n_features = X_train.shape[-1]
    scaler = StandardScaler()
    scaler.fit(X_train.reshape(-1, n_features))
    split_dict["scaler"] = scaler
    split_dict["X_train_scaled"] = scaler.transform(X_train.reshape(-1, n_features)).reshape(X_train.shape)
    split_dict["X_val_scaled"] = scaler.transform(X_val.reshape(-1, n_features)).reshape(X_val.shape)
    split_dict["X_test_scaled"] = scaler.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape)
    split_dict["X_train_flat"] = split_dict["X_train_scaled"].reshape(len(X_train), -1)
    split_dict["X_val_flat"] = split_dict["X_val_scaled"].reshape(len(X_val), -1)
    split_dict["X_test_flat"] = split_dict["X_test_scaled"].reshape(len(X_test), -1)
    return split_dict

class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout):
        super().__init__()
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout)
        self.dropout1 = layers.Dropout(dropout)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn = models.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dropout(dropout),
            layers.Dense(d_model)
        ])
        self.dropout2 = layers.Dropout(dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs, training=False, return_scores=False):
        attn_output, attn_scores = self.mha(inputs, inputs, return_attention_scores=True, training=training)
        x = self.norm1(inputs + self.dropout1(attn_output, training=training))
        ffn_output = self.ffn(x, training=training)
        out = self.norm2(x + self.dropout2(ffn_output, training=training))
        if return_scores:
            return out, attn_scores
        return out

def build_transformer(input_shape, d_model=32, num_heads=2, ff_dim=64, dropout=0.10, dense_units=16):
    inp = layers.Input(shape=input_shape)
    proj = layers.Dense(d_model)(inp)
    pos_idx = tf.range(start=0, limit=input_shape[0], delta=1)
    pos_emb = layers.Embedding(input_dim=input_shape[0], output_dim=d_model)(pos_idx)
    x = proj + pos_emb
    block = TransformerBlock(d_model=d_model, num_heads=num_heads, ff_dim=ff_dim, dropout=dropout)
    x = block(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1)(x)
    return models.Model(inp, out)

def build_gru(input_shape, units=32, dense_units=16, dropout=0.10):
    inp = layers.Input(shape=input_shape)
    x = layers.GRU(units, dropout=dropout, recurrent_dropout=0.0)(inp)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1)(x)
    return models.Model(inp, out)

def benchmark_predict_latency(model_obj, X_sample, model_type, repeats=25):
    if len(X_sample) == 0:
        return np.nan
    if model_type == "keras":
        _ = model_obj(X_sample[:8], training=False).numpy()
        t0 = time.perf_counter()
        for _ in range(repeats):
            _ = model_obj(X_sample, training=False).numpy()
        total = time.perf_counter() - t0
    else:
        _ = model_obj.predict(X_sample[:8])
        t0 = time.perf_counter()
        for _ in range(repeats):
            _ = model_obj.predict(X_sample)
        total = time.perf_counter() - t0
    return (total / repeats) / len(X_sample) * 1000.0

def assign_state_labels(center_df):
    label_map = {}
    if len(center_df) == 1:
        return {center_df.index[0]: "Stable / Reference"}

    temp = center_df.copy()
    use_cols = [c for c in ["mean_do", "mean_survival", "mean_weight"] if c in temp.columns]
    inv_cols = [c for c in ["mean_disease", "mean_turbidity"] if c in temp.columns]

    score_stable = pd.Series(0.0, index=temp.index)
    score_stress = pd.Series(0.0, index=temp.index)

    for c in use_cols:
        rng = temp[c].max() - temp[c].min()
        norm = (temp[c] - temp[c].min()) / (rng if rng != 0 else 1)
        score_stable += norm
        score_stress -= norm

    for c in inv_cols:
        rng = temp[c].max() - temp[c].min()
        norm = (temp[c] - temp[c].min()) / (rng if rng != 0 else 1)
        score_stable -= norm
        score_stress += norm

    stable_id = score_stable.idxmax()
    stress_id = score_stress.drop(index=[stable_id]).idxmax() if len(temp) > 1 else stable_id

    label_map[stable_id] = "Stable / Oxygen-Rich"
    if stress_id not in label_map:
        label_map[stress_id] = "Stress / Degraded"

    remaining = [idx for idx in temp.index if idx not in label_map]
    for i, idx in enumerate(remaining, start=1):
        label_map[idx] = f"Transitional / Mixed {i}"

    return label_map

df = pd.read_excel(FILE_PATH)
df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")
df = df.dropna(subset=["Datetime"]).sort_values("Datetime").reset_index(drop=True)

water_vars = [c for c in ["Temperature (°C)", "Dissolved Oxygen (mg/L)", "pH", "Turbidity (NTU)", "Precipitation (inches)"] if c in df.columns]
tilapia_vars = [c for c in ["Average Fish Weight (g)", "Survival Rate (%)", "Disease Occurrence (Cases)"] if c in df.columns]
health_var = "Health Status" if "Health Status" in df.columns else None

anomaly_vars = [c for c in ["Temperature (°C)", "Dissolved Oxygen (mg/L)", "pH", "Turbidity (NTU)", "Average Fish Weight (g)", "Survival Rate (%)", "Disease Occurrence (Cases)"] if c in df.columns]
for col in anomaly_vars:
    df[f"z__{col}"] = robust_z(df[col])

z_cols = [f"z__{c}" for c in anomaly_vars]
df["anomaly_score"] = df[z_cols].abs().mean(axis=1)
score_threshold = max(2.5, float(df["anomaly_score"].quantile(0.98)))
df["anomaly_flag"] = (df["anomaly_score"] >= score_threshold) | (df[z_cols].abs().max(axis=1) >= 3.0)

flagged = df[df["anomaly_flag"]].copy()
episode_rows = []

if len(flagged) > 0:
    flagged["gap_hours"] = flagged["Datetime"].diff().dt.total_seconds().div(3600).fillna(0)
    flagged["episode_id"] = (flagged["gap_hours"] > 2).cumsum()

    for ep_id, part in flagged.groupby("episode_id"):
        sig_scores = {c: part[f"z__{c}"].abs().mean() for c in anomaly_vars}
        top_sig = sorted(sig_scores.items(), key=lambda x: x[1], reverse=True)[:3]
        top_sig_text = ", ".join([f"{k} ({v:.2f})" for k, v in top_sig])

        episode_rows.append({
            "episode_id": int(ep_id) + 1,
            "start_time": str(part["Datetime"].min()),
            "end_time": str(part["Datetime"].max()),
            "duration_hours": (part["Datetime"].max() - part["Datetime"].min()).total_seconds() / 3600.0 + 1,
            "anomalous_points": int(len(part)),
            "peak_anomaly_score": float(part["anomaly_score"].max()),
            "mean_temperature_c": float(part["Temperature (°C)"].mean()) if "Temperature (°C)" in part.columns else np.nan,
            "mean_do_mg_l": float(part["Dissolved Oxygen (mg/L)"].mean()) if "Dissolved Oxygen (mg/L)" in part.columns else np.nan,
            "mean_pH": float(part["pH"].mean()) if "pH" in part.columns else np.nan,
            "mean_turbidity_ntu": float(part["Turbidity (NTU)"].mean()) if "Turbidity (NTU)" in part.columns else np.nan,
            "mean_survival_pct": float(part["Survival Rate (%)"].mean()) if "Survival Rate (%)" in part.columns else np.nan,
            "mean_disease_cases": float(part["Disease Occurrence (Cases)"].mean()) if "Disease Occurrence (Cases)" in part.columns else np.nan,
            "dominant_health_status": part[health_var].mode().iloc[0] if health_var is not None and len(part[health_var].mode()) > 0 else "",
            "variable_signature": top_sig_text
        })

table01 = pd.DataFrame(episode_rows)
if len(table01) == 0:
    fallback = df.nlargest(8, "anomaly_score").copy()
    table01 = pd.DataFrame({
        "episode_id": np.arange(1, len(fallback) + 1),
        "start_time": fallback["Datetime"].astype(str).values,
        "end_time": fallback["Datetime"].astype(str).values,
        "duration_hours": 1.0,
        "anomalous_points": 1,
        "peak_anomaly_score": fallback["anomaly_score"].values,
        "mean_temperature_c": fallback["Temperature (°C)"].values if "Temperature (°C)" in fallback.columns else np.nan,
        "mean_do_mg_l": fallback["Dissolved Oxygen (mg/L)"].values if "Dissolved Oxygen (mg/L)" in fallback.columns else np.nan,
        "mean_pH": fallback["pH"].values if "pH" in fallback.columns else np.nan,
        "mean_turbidity_ntu": fallback["Turbidity (NTU)"].values if "Turbidity (NTU)" in fallback.columns else np.nan,
        "mean_survival_pct": fallback["Survival Rate (%)"].values if "Survival Rate (%)" in fallback.columns else np.nan,
        "mean_disease_cases": fallback["Disease Occurrence (Cases)"].values if "Disease Occurrence (Cases)" in fallback.columns else np.nan,
        "dominant_health_status": fallback[health_var].astype(str).values if health_var is not None else "",
        "variable_signature": ["Top instantaneous anomaly"] * len(fallback)
    })
table01 = table01.sort_values(["peak_anomaly_score", "duration_hours"], ascending=[False, False]).head(10).reset_index(drop=True)
save_csv(table01, "table01_anomaly_case_summary.csv")

state_vars = [c for c in ["Temperature (°C)", "Dissolved Oxygen (mg/L)", "pH", "Turbidity (NTU)", "Average Fish Weight (g)", "Survival Rate (%)", "Disease Occurrence (Cases)"] if c in df.columns]
state_frame = df[["Datetime"] + state_vars + ["anomaly_flag"]].dropna().reset_index(drop=True)
state_scaler = StandardScaler()
state_X = state_scaler.fit_transform(state_frame[state_vars])

k_scores = []
for k in [2, 3, 4]:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(state_X)
    score = silhouette_score(state_X, labels)
    k_scores.append((k, score))
best_k = sorted(k_scores, key=lambda x: x[1], reverse=True)[0][0]

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
state_frame["cluster_id"] = kmeans.fit_predict(state_X)

centers_scaled = pd.DataFrame(kmeans.cluster_centers_, columns=state_vars)
centers = pd.DataFrame(state_scaler.inverse_transform(centers_scaled), columns=state_vars)
center_summary = pd.DataFrame(index=centers.index)
center_summary["mean_do"] = centers["Dissolved Oxygen (mg/L)"] if "Dissolved Oxygen (mg/L)" in centers.columns else 0
center_summary["mean_survival"] = centers["Survival Rate (%)"] if "Survival Rate (%)" in centers.columns else 0
center_summary["mean_weight"] = centers["Average Fish Weight (g)"] if "Average Fish Weight (g)" in centers.columns else 0
center_summary["mean_disease"] = centers["Disease Occurrence (Cases)"] if "Disease Occurrence (Cases)" in centers.columns else 0
center_summary["mean_turbidity"] = centers["Turbidity (NTU)"] if "Turbidity (NTU)" in centers.columns else 0

state_label_map = assign_state_labels(center_summary)
state_frame["state_label"] = state_frame["cluster_id"].map(state_label_map)

cluster_rows = []
for cid, part in state_frame.groupby("cluster_id"):
    cluster_rows.append({
        "cluster_id": int(cid),
        "state_label": state_label_map[cid],
        "sample_count": int(len(part)),
        "sample_fraction_pct": 100 * len(part) / len(state_frame),
        "anomaly_rate_pct": 100 * part["anomaly_flag"].mean(),
        "mean_temperature_c": float(part["Temperature (°C)"].mean()) if "Temperature (°C)" in part.columns else np.nan,
        "mean_do_mg_l": float(part["Dissolved Oxygen (mg/L)"].mean()) if "Dissolved Oxygen (mg/L)" in part.columns else np.nan,
        "mean_pH": float(part["pH"].mean()) if "pH" in part.columns else np.nan,
        "mean_turbidity_ntu": float(part["Turbidity (NTU)"].mean()) if "Turbidity (NTU)" in part.columns else np.nan,
        "mean_weight_g": float(part["Average Fish Weight (g)"].mean()) if "Average Fish Weight (g)" in part.columns else np.nan,
        "mean_survival_pct": float(part["Survival Rate (%)"].mean()) if "Survival Rate (%)" in part.columns else np.nan,
        "mean_disease_cases": float(part["Disease Occurrence (Cases)"].mean()) if "Disease Occurrence (Cases)" in part.columns else np.nan,
        "characteristic_signature": ""
    })

table03 = pd.DataFrame(cluster_rows).sort_values("mean_survival_pct", ascending=False).reset_index(drop=True)

sig_texts = []
for _, row in table03.iterrows():
    comps = []
    if pd.notna(row["mean_do_mg_l"]):
        comps.append(f"DO={row['mean_do_mg_l']:.2f}")
    if pd.notna(row["mean_turbidity_ntu"]):
        comps.append(f"Turbidity={row['mean_turbidity_ntu']:.2f}")
    if pd.notna(row["mean_survival_pct"]):
        comps.append(f"Survival={row['mean_survival_pct']:.2f}")
    if pd.notna(row["mean_disease_cases"]):
        comps.append(f"Disease={row['mean_disease_cases']:.2f}")
    sig_texts.append(", ".join(comps))
table03["characteristic_signature"] = sig_texts
save_csv(table03, "table03_cluster_or_state_summary.csv")

df = df.merge(state_frame[["Datetime", "cluster_id", "state_label"]], on="Datetime", how="left")

pattern_rows = []

for var in [c for c in ["Dissolved Oxygen (mg/L)", "Temperature (°C)", "Turbidity (NTU)", "pH"] if c in df.columns]:
    temp = df.copy()
    temp["condition_level"] = safe_quantile_groups(temp[var], q=3)
    g = temp.groupby("condition_level", dropna=False).agg(
        sample_count=("condition_level", "size"),
        survival_mean=("Survival Rate (%)", "mean") if "Survival Rate (%)" in temp.columns else ("condition_level", "size"),
        disease_mean=("Disease Occurrence (Cases)", "mean") if "Disease Occurrence (Cases)" in temp.columns else ("condition_level", "size"),
        weight_mean=("Average Fish Weight (g)", "mean") if "Average Fish Weight (g)" in temp.columns else ("condition_level", "size")
    ).reset_index()
    for _, r in g.iterrows():
        if pd.isna(r["condition_level"]):
            continue
        interpretation = ""
        if var == "Dissolved Oxygen (mg/L)" and str(r["condition_level"]) == "Low":
            interpretation = "Low dissolved oxygen corresponds to a more stressed fish-health regime."
        elif var == "Dissolved Oxygen (mg/L)" and str(r["condition_level"]) == "High":
            interpretation = "Higher dissolved oxygen aligns with improved stability and survival."
        elif var == "Turbidity (NTU)" and str(r["condition_level"]) == "High":
            interpretation = "High turbidity aligns with poorer water clarity and elevated biological stress."
        elif var == "Temperature (°C)" and str(r["condition_level"]) != "Medium":
            interpretation = "Temperature deviations from the central band coincide with less stable conditions."
        elif var == "pH":
            interpretation = "pH bands show narrower effects than oxygen and turbidity but still shift outcomes."
        pattern_rows.append({
            "pattern_type": "Single-variable condition",
            "condition": f"{var} = {r['condition_level']}",
            "support_pct": 100 * r["sample_count"] / len(temp),
            "mean_survival_pct": float(r["survival_mean"]) if "survival_mean" in r else np.nan,
            "mean_disease_cases": float(r["disease_mean"]) if "disease_mean" in r else np.nan,
            "mean_weight_g": float(r["weight_mean"]) if "weight_mean" in r else np.nan,
            "interpretation": interpretation
        })

if {"Dissolved Oxygen (mg/L)", "Turbidity (NTU)"}.issubset(df.columns):
    temp = df.copy()
    temp["do_level"] = safe_quantile_groups(temp["Dissolved Oxygen (mg/L)"], q=3)
    temp["turb_level"] = safe_quantile_groups(temp["Turbidity (NTU)"], q=3)

    mask_combo = (temp["do_level"].astype(str) == "Low") & (temp["turb_level"].astype(str) == "High")
    combo = temp[mask_combo]
    if len(combo) > 0:
        pattern_rows.append({
            "pattern_type": "Combined condition",
            "condition": "Low dissolved oxygen + High turbidity",
            "support_pct": 100 * len(combo) / len(temp),
            "mean_survival_pct": float(combo["Survival Rate (%)"].mean()) if "Survival Rate (%)" in combo.columns else np.nan,
            "mean_disease_cases": float(combo["Disease Occurrence (Cases)"].mean()) if "Disease Occurrence (Cases)" in combo.columns else np.nan,
            "mean_weight_g": float(combo["Average Fish Weight (g)"].mean()) if "Average Fish Weight (g)" in combo.columns else np.nan,
            "interpretation": "The combined low-oxygen and high-turbidity regime captures the clearest degraded-condition signature."
        })

if "state_label" in df.columns:
    g = df.groupby("state_label").agg(
        sample_count=("state_label", "size"),
        survival_mean=("Survival Rate (%)", "mean") if "Survival Rate (%)" in df.columns else ("state_label", "size"),
        disease_mean=("Disease Occurrence (Cases)", "mean") if "Disease Occurrence (Cases)" in df.columns else ("state_label", "size"),
        weight_mean=("Average Fish Weight (g)", "mean") if "Average Fish Weight (g)" in df.columns else ("state_label", "size")
    ).reset_index()
    for _, r in g.iterrows():
        pattern_rows.append({
            "pattern_type": "Discovered state",
            "condition": f"State = {r['state_label']}",
            "support_pct": 100 * r["sample_count"] / len(df),
            "mean_survival_pct": float(r["survival_mean"]),
            "mean_disease_cases": float(r["disease_mean"]),
            "mean_weight_g": float(r["weight_mean"]),
            "interpretation": "The discovered state summarizes a recurring multimodal operating condition."
        })

table02 = pd.DataFrame(pattern_rows)
table02["priority_score"] = table02["support_pct"] * (table02["mean_survival_pct"].fillna(table02["mean_survival_pct"].median()) if "mean_survival_pct" in table02.columns else 1)
table02 = table02.sort_values(["pattern_type", "support_pct"], ascending=[True, False]).drop(columns=["priority_score"]).reset_index(drop=True)
table02 = table02.head(12)
save_csv(table02, "table02_discovered_patterns_and_rules.csv")

phase4_table_path = PHASE4_DIR / "table01_overall_model_comparison.csv"
if phase4_table_path.exists():
    phase4_cmp = pd.read_csv(phase4_table_path)
else:
    phase4_cmp = pd.DataFrame(columns=["model_name", "MAE", "RMSE", "R2", "MAPE", "Bias", "train_seconds", "params", "approx_size_mb"])

work = build_feature_frame(df)
target_col = "Survival Rate (%)"
seq_len = 24
horizon = 1

raw_sensor_features = [c for c in ["Temperature (°C)", "Dissolved Oxygen (mg/L)", "pH", "Turbidity (NTU)", "Precipitation (inches)"] if c in work.columns]
historical_outcome_features = [c for c in ["Average Fish Weight (g)", "Disease Occurrence (Cases)", "Survival Rate (%)"] if c in work.columns]
control_features = [c for c in ["Oxygenation Interventions", "Corrective Interventions", "Oxygenation_Automatic_bin", "Corrective_Measures_bin", "Thermal_Risk_bin"] if c in work.columns]
calendar_features = [c for c in ["hour_sin", "hour_cos", "month_sin", "month_cos"] if c in work.columns]
interaction_features = [c for c in ["temp_x_do", "do_to_temp_ratio", "temp_x_turbidity", "do_x_ph"] if c in work.columns]
engineered_features = [c for c in work.columns if any(tag in c for tag in ["__rollmean_", "__rollstd_", "__trend_"])]

full_feature_cols = list(dict.fromkeys(raw_sensor_features + historical_outcome_features + control_features + calendar_features + interaction_features + engineered_features))
X_full, y_full, times_full, rows_full = build_sequences(work, full_feature_cols, target_col, seq_len=seq_len, horizon=horizon)
split_full = scale_splits(chronological_split(X_full, y_full, times_full, rows_full))

X_train_flat = split_full["X_train_flat"]
X_test_flat = split_full["X_test_flat"]
X_train_seq = split_full["X_train_scaled"]
X_test_seq = split_full["X_test_scaled"]

sample_n = min(128, len(X_test_flat))
X_sample_flat = X_test_flat[:sample_n]
X_sample_seq = X_test_seq[:sample_n]

edge_rows = []

set_seed(42)

linreg = LinearRegression()
linreg.fit(X_train_flat, split_full["y_train"])
lin_latency = benchmark_predict_latency(linreg, X_sample_flat, model_type="sklearn", repeats=40)
lin_size = len(pickle.dumps(linreg)) / (1024 ** 2)
edge_rows.append({"model_name": "Linear Regression", "latency_ms_per_sample": lin_latency, "benchmark_size_mb": lin_size})

rf = RandomForestRegressor(
    n_estimators=160,
    max_depth=12,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_flat, split_full["y_train"])
rf_latency = benchmark_predict_latency(rf, X_sample_flat, model_type="sklearn", repeats=20)
rf_size = len(pickle.dumps(rf)) / (1024 ** 2)
edge_rows.append({"model_name": "Random Forest", "latency_ms_per_sample": rf_latency, "benchmark_size_mb": rf_size})

xgb_model_name = "XGBoost"
try:
    from xgboost import XGBRegressor
    xgb = XGBRegressor(
        n_estimators=220,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
    xgb.fit(X_train_flat, split_full["y_train"], verbose=False)
    xgb_latency = benchmark_predict_latency(xgb, X_sample_flat, model_type="sklearn", repeats=20)
    xgb_size = len(pickle.dumps(xgb)) / (1024 ** 2)
except Exception:
    xgb_model_name = "HistGradientBoosting"
    xgb = HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=4,
        max_iter=220,
        random_state=42
    )
    xgb.fit(X_train_flat, split_full["y_train"])
    xgb_latency = benchmark_predict_latency(xgb, X_sample_flat, model_type="sklearn", repeats=20)
    xgb_size = len(pickle.dumps(xgb)) / (1024 ** 2)
edge_rows.append({"model_name": xgb_model_name, "latency_ms_per_sample": xgb_latency, "benchmark_size_mb": xgb_size})

gru_model = build_gru(input_shape=(X_train_seq.shape[1], X_train_seq.shape[2]))
gru_latency = benchmark_predict_latency(gru_model, X_sample_seq, model_type="keras", repeats=30)
gru_size = gru_model.count_params() * 4 / (1024 ** 2)
edge_rows.append({"model_name": "GRU", "latency_ms_per_sample": gru_latency, "benchmark_size_mb": gru_size})

transformer_model = build_transformer(input_shape=(X_train_seq.shape[1], X_train_seq.shape[2]))
transformer_latency = benchmark_predict_latency(transformer_model, X_sample_seq, model_type="keras", repeats=30)
transformer_size = transformer_model.count_params() * 4 / (1024 ** 2)
edge_rows.append({"model_name": "Lightweight Transformer", "latency_ms_per_sample": transformer_latency, "benchmark_size_mb": transformer_size})

edge_bench = pd.DataFrame(edge_rows)

table04 = phase4_cmp.merge(edge_bench, on="model_name", how="outer")
table04["model_size_mb"] = table04["approx_size_mb"].fillna(table04["benchmark_size_mb"])
table04["throughput_samples_per_sec"] = 1000.0 / table04["latency_ms_per_sample"]

rmse_min, rmse_max = table04["RMSE"].min(), table04["RMSE"].max()
lat_min, lat_max = table04["latency_ms_per_sample"].min(), table04["latency_ms_per_sample"].max()
size_min, size_max = table04["model_size_mb"].min(), table04["model_size_mb"].max()

def norm_col(s, smin, smax):
    if pd.isna(smin) or pd.isna(smax) or smax == smin:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - smin) / (smax - smin)

table04["rmse_norm"] = norm_col(table04["RMSE"], rmse_min, rmse_max)
table04["lat_norm"] = norm_col(table04["latency_ms_per_sample"], lat_min, lat_max)
table04["size_norm"] = norm_col(table04["model_size_mb"], size_min, size_max)
table04["edge_efficiency_score"] = 1 - (0.50 * table04["rmse_norm"] + 0.30 * table04["lat_norm"] + 0.20 * table04["size_norm"])

table04["deployment_note"] = ""
for i in table04.index:
    name = table04.loc[i, "model_name"]
    rmse = table04.loc[i, "RMSE"]
    size = table04.loc[i, "model_size_mb"]
    lat = table04.loc[i, "latency_ms_per_sample"]
    if name == "Lightweight Transformer":
        table04.loc[i, "deployment_note"] = "Very compact architecture; edge feasibility depends on closing the current accuracy gap."
    elif name == "GRU":
        table04.loc[i, "deployment_note"] = "Compact recurrent model, but unstable accuracy weakens deployment confidence."
    elif name in ["XGBoost", "HistGradientBoosting"]:
        table04.loc[i, "deployment_note"] = "Best predictive accuracy in this experiment; tree complexity should be checked against memory budget."
    elif name == "Random Forest":
        table04.loc[i, "deployment_note"] = "Robust baseline but ensemble size can limit strict edge budgets."
    elif name == "Linear Regression":
        table04.loc[i, "deployment_note"] = "Very light deployment footprint but limited predictive fidelity."
    else:
        table04.loc[i, "deployment_note"] = "Reference model."

table04 = table04[[
    "model_name", "RMSE", "MAE", "latency_ms_per_sample", "throughput_samples_per_sec",
    "model_size_mb", "params", "train_seconds", "edge_efficiency_score", "deployment_note"
]].sort_values(["edge_efficiency_score", "RMSE"], ascending=[False, True]).reset_index(drop=True)
save_csv(table04, "table04_edge_deployment_summary.csv")

best_model_row = phase4_cmp.sort_values("RMSE").iloc[0] if len(phase4_cmp) > 0 else table04.sort_values("RMSE").iloc[0]
best_model_name = best_model_row["model_name"]

stable_state_row = table03.sort_values("mean_survival_pct", ascending=False).iloc[0] if len(table03) > 0 else None
stress_state_row = table03.sort_values(["mean_survival_pct", "mean_disease_cases"], ascending=[True, False]).iloc[0] if len(table03) > 0 else None
top_anomaly_row = table01.sort_values("peak_anomaly_score", ascending=False).iloc[0] if len(table01) > 0 else None
low_do_row = table02[table02["condition"].astype(str).str.contains("Dissolved Oxygen", na=False) & table02["condition"].astype(str).str.contains("Low", na=False)]
high_do_row = table02[table02["condition"].astype(str).str.contains("Dissolved Oxygen", na=False) & table02["condition"].astype(str).str.contains("High", na=False)]
combo_row = table02[table02["condition"].astype(str).str.contains("Low dissolved oxygen + High turbidity", na=False)]
transformer_row = table04[table04["model_name"] == "Lightweight Transformer"].iloc[0] if "Lightweight Transformer" in table04["model_name"].values else None

takeaways = []

if stable_state_row is not None:
    takeaways.append({
        "theme": "Discovered stable operating regime",
        "finding": f"{stable_state_row['state_label']} is the most favorable recurring state with mean survival {stable_state_row['mean_survival_pct']:.2f}% and dissolved oxygen {stable_state_row['mean_do_mg_l']:.2f} mg/L.",
        "implication": "Oxygen-rich and clearer-water conditions define the most biologically stable regime in this dataset.",
        "practical_recommendation": "Maintain dissolved oxygen near the upper observed regime and avoid drift toward the stressed state signature.",
        "evidence_basis": "Cluster/state summary"
    })

if len(low_do_row) > 0 and len(high_do_row) > 0:
    ld = low_do_row.iloc[0]
    hd = high_do_row.iloc[0]
    takeaways.append({
        "theme": "Oxygen-linked outcome gradient",
        "finding": f"Moving from low to high dissolved oxygen shifts mean survival from {ld['mean_survival_pct']:.2f}% to {hd['mean_survival_pct']:.2f}% and reduces disease burden from {ld['mean_disease_cases']:.2f} to {hd['mean_disease_cases']:.2f}.",
        "implication": "Dissolved oxygen is the clearest controllable lever for healthier tilapia conditions in this study.",
        "practical_recommendation": "Use oxygen monitoring and timely aeration as the primary edge-alert trigger.",
        "evidence_basis": "Discovered patterns and rules"
    })

if len(combo_row) > 0:
    cr = combo_row.iloc[0]
    takeaways.append({
        "theme": "Compound stress regime",
        "finding": f"The low-oxygen and high-turbidity combination yields mean survival {cr['mean_survival_pct']:.2f}% with disease level {cr['mean_disease_cases']:.2f}.",
        "implication": "Compound conditions matter more than single-variable deviations for identifying biologically stressed periods.",
        "practical_recommendation": "Escalate alerts when low dissolved oxygen co-occurs with elevated turbidity instead of relying on single-sensor alarms alone.",
        "evidence_basis": "Combined rule analysis"
    })

if top_anomaly_row is not None:
    takeaways.append({
        "theme": "Anomaly-aware monitoring",
        "finding": f"The most abnormal period started on {top_anomaly_row['start_time']} with signature {top_anomaly_row['variable_signature']}.",
        "implication": "Anomaly scoring helps compress long sensor streams into actionable episodes with clear variable signatures.",
        "practical_recommendation": "Review anomaly episodes rather than raw logs, and prioritize signatures dominated by oxygen, turbidity, or disease shifts.",
        "evidence_basis": "Anomaly case summary"
    })

if transformer_row is not None:
    best_edge = table04.sort_values(["edge_efficiency_score", "RMSE"], ascending=[False, True]).iloc[0]
    takeaways.append({
        "theme": "Edge-intelligence deployment judgment",
        "finding": f"The current best predictive model is {best_model_name}, while the Lightweight Transformer remains extremely compact at about {transformer_row['model_size_mb']:.4f} MB but with weaker accuracy than the best baseline.",
        "implication": "Model compactness alone is not enough to justify edge deployment claims; accuracy-efficiency balance must be demonstrated jointly.",
        "practical_recommendation": f"Use {best_model_name} as the current accuracy reference and treat the Transformer as a promising but not yet dominant edge model in this formulation.",
        "evidence_basis": "Model comparison and edge deployment summary"
    })

table05 = pd.DataFrame(takeaways).head(5)
save_csv(table05, "table05_final_scientific_takeaways.csv")

fig, axes = plt.subplots(2, 1, figsize=(13, 8), constrained_layout=True)
axes[0].plot(df["Datetime"], df["anomaly_score"], linewidth=1.2)
axes[0].axhline(score_threshold, linestyle="--", linewidth=1.0)
axes[0].scatter(df.loc[df["anomaly_flag"], "Datetime"], df.loc[df["anomaly_flag"], "anomaly_score"], s=12)
axes[0].set_title("Anomaly Score Timeline")
axes[0].set_ylabel("Anomaly score")
axes[0].grid(alpha=0.2)

if "Dissolved Oxygen (mg/L)" in df.columns and "Survival Rate (%)" in df.columns:
    do_norm = (df["Dissolved Oxygen (mg/L)"] - df["Dissolved Oxygen (mg/L)"].mean()) / (df["Dissolved Oxygen (mg/L)"].std() if df["Dissolved Oxygen (mg/L)"].std() != 0 else 1)
    surv_norm = (df["Survival Rate (%)"] - df["Survival Rate (%)"].mean()) / (df["Survival Rate (%)"].std() if df["Survival Rate (%)"].std() != 0 else 1)
    axes[1].plot(df["Datetime"], do_norm, linewidth=1.0, label="Dissolved Oxygen (z)")
    axes[1].plot(df["Datetime"], surv_norm, linewidth=1.0, label="Survival Rate (z)")
    axes[1].scatter(df.loc[df["anomaly_flag"], "Datetime"], surv_norm[df["anomaly_flag"]], s=10)
    axes[1].set_title("Key Biological Signals During Anomalous Periods")
    axes[1].set_ylabel("Standardized value")
    axes[1].legend(frameon=False)
    axes[1].grid(alpha=0.2)
else:
    axes[1].plot(df["Datetime"], df["anomaly_score"], linewidth=1.0)
    axes[1].set_title("Anomalous Period Emphasis")
    axes[1].grid(alpha=0.2)

fig.suptitle("Timeline of Detected Anomalies and Unusual Events", fontsize=13, fontweight="bold")
save_figure(fig, "fig01_anomaly_timeline.png")

pca = PCA(n_components=2, random_state=42)
state_xy = pca.fit_transform(state_X)
centers_xy = pca.transform(kmeans.cluster_centers_)

fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)
for cid in sorted(state_frame["cluster_id"].unique()):
    mask = state_frame["cluster_id"] == cid
    ax.scatter(state_xy[mask, 0], state_xy[mask, 1], s=14, alpha=0.55, label=state_label_map[cid])

ax.scatter(centers_xy[:, 0], centers_xy[:, 1], s=140, marker="X", edgecolors="black", linewidths=1.0)
for cid, (xv, yv) in enumerate(centers_xy):
    ax.text(xv, yv, state_label_map[cid], fontsize=8, ha="left", va="bottom")

ax.set_title(f"Discovered Scientific Operating States (k={best_k})")
ax.set_xlabel("Principal component 1")
ax.set_ylabel("Principal component 2")
ax.legend(frameon=False, loc="best")
ax.grid(alpha=0.2)
save_figure(fig, "fig02_cluster_or_state_map.png")

do_levels = safe_quantile_groups(df["Dissolved Oxygen (mg/L)"], q=3) if "Dissolved Oxygen (mg/L)" in df.columns else None
turb_levels = safe_quantile_groups(df["Turbidity (NTU)"], q=3) if "Turbidity (NTU)" in df.columns else None
temp_levels = safe_quantile_groups(df["Temperature (°C)"], q=3) if "Temperature (°C)" in df.columns else None

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6), constrained_layout=True)

if do_levels is not None:
    temp_df = pd.DataFrame({"level": do_levels.astype(str), "survival": df["Survival Rate (%)"]})
    order = [x for x in ["Low", "Medium", "High", "All"] if x in temp_df["level"].unique()]
    g = temp_df.groupby("level")["survival"].mean().reindex(order)
    axes[0].bar(g.index, g.values)
    axes[0].set_title("Survival Across Dissolved-Oxygen Bands")
    axes[0].set_xlabel("DO condition")
    axes[0].set_ylabel("Mean survival (%)")
    axes[0].grid(axis="y", alpha=0.2)
else:
    axes[0].axis("off")

if turb_levels is not None:
    temp_df = pd.DataFrame({"level": turb_levels.astype(str), "disease": df["Disease Occurrence (Cases)"]})
    order = [x for x in ["Low", "Medium", "High", "All"] if x in temp_df["level"].unique()]
    g = temp_df.groupby("level")["disease"].mean().reindex(order)
    axes[1].bar(g.index, g.values)
    axes[1].set_title("Disease Burden Across Turbidity Bands")
    axes[1].set_xlabel("Turbidity condition")
    axes[1].set_ylabel("Mean disease cases")
    axes[1].grid(axis="y", alpha=0.2)
else:
    axes[1].axis("off")

if do_levels is not None and turb_levels is not None:
    temp = pd.DataFrame({
        "do_level": do_levels.astype(str),
        "turb_level": turb_levels.astype(str),
        "survival": df["Survival Rate (%)"]
    })
    rows = [x for x in ["Low", "Medium", "High", "All"] if x in temp["do_level"].unique()]
    cols = [x for x in ["Low", "Medium", "High", "All"] if x in temp["turb_level"].unique()]
    pivot = temp.pivot_table(index="do_level", columns="turb_level", values="survival", aggfunc="mean").reindex(index=rows, columns=cols)
    im = axes[2].imshow(pivot.values, aspect="auto")
    axes[2].set_xticks(np.arange(len(cols)))
    axes[2].set_xticklabels(cols)
    axes[2].set_yticks(np.arange(len(rows)))
    axes[2].set_yticklabels(rows)
    axes[2].set_title("Mean Survival: DO × Turbidity")
    axes[2].set_xlabel("Turbidity band")
    axes[2].set_ylabel("DO band")
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.iloc[i, j]
            if pd.notna(val):
                axes[2].text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8)
    cbar = fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
    cbar.set_label("Mean survival (%)")
else:
    axes[2].axis("off")

fig.suptitle("Interpretable Water-Quality Patterns Linked to Tilapia Outcomes", fontsize=13, fontweight="bold")
save_figure(fig, "fig03_rule_or_pattern_summary.png")

edge_plot = table04.copy().dropna(subset=["RMSE", "latency_ms_per_sample", "model_size_mb"])
fig, ax = plt.subplots(figsize=(9, 6.5), constrained_layout=True)
bubble_sizes = 300 + 900 * (edge_plot["model_size_mb"] / edge_plot["model_size_mb"].max().replace(0, 1) if hasattr(edge_plot["model_size_mb"].max(), "replace") else edge_plot["model_size_mb"] / max(edge_plot["model_size_mb"].max(), 1e-6))
ax.scatter(edge_plot["latency_ms_per_sample"], edge_plot["RMSE"], s=bubble_sizes, alpha=0.6)
for _, r in edge_plot.iterrows():
    ax.text(r["latency_ms_per_sample"], r["RMSE"], r["model_name"], fontsize=8, ha="left", va="bottom")
ax.set_xlabel("Inference latency (ms per sample)")
ax.set_ylabel("RMSE on test set")
ax.set_title("Accuracy–Efficiency Trade-Off Across Candidate Models")
ax.grid(alpha=0.2)
ax.set_xscale("log")
save_figure(fig, "fig04_edge_efficiency_tradeoff.png")

fig, ax = plt.subplots(figsize=(14, 7), constrained_layout=True)
ax.axis("off")

box_specs = [
    (0.03, 0.62, 0.17, 0.22, "IoT Sensing\nand Data Integrity", "Water quality\nTilapia outcomes\nTimestamped streams"),
    (0.23, 0.62, 0.17, 0.22, "Multimodal\nExploratory Analysis", "Temporal trends\nCorrelations\nSeasonal structure"),
    (0.43, 0.62, 0.17, 0.22, "Temporal\nFeature Structuring", "Lags\nRolling signals\nInteraction features"),
    (0.63, 0.62, 0.17, 0.22, "Lightweight\nPredictive Modeling", "Edge-aware Transformer\nBaselines\nControlled comparison"),
    (0.83, 0.62, 0.14, 0.22, "Knowledge Discovery\nand Edge Review", "Anomalies\nStates\nRules\nDeployment judgment"),
]

for x, y, w, h, title, subtitle in box_specs:
    rect = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.02", linewidth=1.2, facecolor="white")
    ax.add_patch(rect)
    ax.text(x + w/2, y + h*0.68, title, ha="center", va="center", fontsize=11, fontweight="bold")
    ax.text(x + w/2, y + h*0.28, subtitle, ha="center", va="center", fontsize=9)

for x1, x2 in [(0.20, 0.23), (0.40, 0.43), (0.60, 0.63), (0.80, 0.83)]:
    ax.annotate("", xy=(x2, 0.73), xytext=(x1, 0.73), arrowprops=dict(arrowstyle="->", lw=1.5))

bottom_boxes = [
    (0.08, 0.20, 0.23, 0.18, "Scientific insight", "Oxygen and turbidity dominate the clearest health-linked gradients."),
    (0.39, 0.20, 0.23, 0.18, "Anomaly-aware monitoring", "Abnormal periods can be compressed into interpretable signatures instead of raw logs."),
    (0.70, 0.20, 0.23, 0.18, "Edge deployment conclusion", "Compact Transformer is promising in size, but current formulation does not yet outperform the strongest tree baseline."),
]

for x, y, w, h, title, subtitle in bottom_boxes:
    rect = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.02", linewidth=1.2, facecolor="white")
    ax.add_patch(rect)
    ax.text(x + w/2, y + h*0.63, title, ha="center", va="center", fontsize=10.5, fontweight="bold")
    ax.text(x + w/2, y + h*0.25, subtitle, ha="center", va="center", fontsize=8.8)

ax.text(0.5, 0.93, "End-to-End Scientific Edge-Intelligence Workflow", ha="center", va="center", fontsize=14, fontweight="bold")
ax.text(0.5, 0.07, "Evidence-based message: compact models are valuable for edge settings, but deployment claims must balance predictive quality with latency and memory constraints.", ha="center", va="center", fontsize=9)
save_figure(fig, "fig05_final_framework_summary.png")

